# Grammar Scoring Engine — spoken English, continuous MOS

Predicts a human MOS grammar score from a 45–60 s `.wav`. Metric: RMSE.
**Settings:** Accelerator = GPU T4 · Internet = On · full run about 45 min with v4's caches attached (§1), about 2 h cold

| stage (LightGBM unless noted) | OOF RMSE |
|---|---|
| baseline (predict the training mean) | 1.01 |
| `repair_wer` alone | 0.84 |
| fluency group (6 features) | 0.95 |
| text group (12 features) | 0.80 |
| 21 handcrafted features | 0.68 |
| + CoLA + sentence embedding | 0.64 |
| + speech SSL embedding (WavLM) | 0.56 |
| + fine-tuned DeBERTa | 0.55 |
| v3 final blend — 9 families × 3 seeds + learned columns | 0.51 |
| v3 public leaderboard | 0.386 |
| v4 public leaderboard — more speech encoders, fine-tuned WavLM | 0.375 |
| **v5 final blend** — five frozen speech encoders, fine-tuned WavLM, speaker neighbours | **TBD** |
| **v5 public leaderboard** | **TBD** |

v3 final blend: Pearson 0.86 · Spearman 0.85 · 70% of predictions within 0.5 MOS · MAE 0.39.
Training RMSE 0.28 — required by the brief. It comes from a LightGBM refit on all rows (§9) and measures memorisation; the out-of-fold number is the result.

The trajectory is the argument of this notebook. Hand-built features took the model from 1.01 to 0.68; every gain after that came from learned representations — first of the transcript, then of the audio itself. The largest came from a speech model that never sees the transcript (§6), which is also the notebook's most important caveat (§10).

**Versions 4 and 5 follow that evidence**: five frozen speech encoders (§6), a fine-tuned speech model (§6b) and a speaker-neighbour column (§6c) — each kept only if it improves the stacked CV.

### The design decision that matters

Whisper is trained on subtitles. Its decoder repairs subject–verb agreement, completes fragments and deletes "um"/"uh" — so features built from a Whisper transcript measure Whisper, not the speaker. Every file is therefore transcribed **twice**: Whisper for word timings, punctuation and clean text; a CTC model, which has no language model, for the uncorrected word sequence. The **edit distance between the two** — how much Whisper had to rewrite the speaker — is the strongest handcrafted feature (Spearman −0.55, monotonic across six bins: 4.48 → 4.05 → 3.39 → 3.14 → 3.00 → 2.83).

### Map

1 setup · 2–3 two ASR passes · 4 handcrafted features · 5 grammar-aware text features · 6 frozen speech encoders · 6b fine-tuned speech models · 6c speaker neighbours · 7 fine-tuned transformer · 8 data checks and adversarial validation · 9 modelling, interpretation and per-candidate report cards · 10 findings.

Every expensive stage caches to `/kaggle/working`. Attach a previous version of this notebook as input (Add Input → Your Work) and §1 copies its caches in, so unchanged stages are skipped.

## 1 · Setup

In [ ]:
import importlib.util, subprocess, sys

for pkg, mod in [("faster-whisper", "faster_whisper"), ("shap", "shap"),
                 ("sentencepiece", "sentencepiece")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

In [ ]:
# =============================================================================
# 1 · Setup — paths, CSVs, audio loading, cache helpers. Safe to re-run.
# On a fresh run no caches exist yet; the cells below build them.
# =============================================================================

import json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()          # silence per-load weight reports
except Exception:
    pass

import os, fnmatch
from collections import Counter

SR         = 16_000
INPUT_ROOT = Path("/kaggle/input")
OUT        = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)


def walk_files(root, patterns):
    """Every file under root matching any pattern. os.walk with followlinks,
    because Path.rglob skips symlinked folders — and Kaggle can mount inputs
    that way, depending on how they were added."""
    if not Path(root).exists():
        return
    for d, _, files in os.walk(root, followlinks=True):
        for f in files:
            if any(fnmatch.fnmatch(f.lower(), p) for p in patterns):
                yield Path(d) / f


# Find the competition files wherever this notebook's inputs are mounted.
csv_paths = list(walk_files(INPUT_ROOT, ("*.csv",)))
pick_csv = lambda name: min((p for p in csv_paths if p.stem.lower().startswith(name)),
                            key=lambda p: (p.stem.lower() != name, len(p.parts)), default=None)
wav_dirs = Counter(p.parent for p in walk_files(INPUT_ROOT, ("*.wav",)))
pick_dir = lambda name: max(((n, d) for d, n in wav_dirs.items() if d.name.lower() == name),
                            default=(0, None))[1]
TRAIN_CSV, TEST_CSV = pick_csv("train"), pick_csv("test")
AUDIO = {"train": pick_dir("train"), "test": pick_dir("test")}

if None in (TRAIN_CSV, TEST_CSV, AUDIO["train"], AUDIO["test"]):
    print("Could not find the competition files. Attached under /kaggle/input:")
    if INPUT_ROOT.exists():
        for d, dirs, files in os.walk(INPUT_ROOT, followlinks=True):
            depth = len(Path(d).relative_to(INPUT_ROOT).parts)
            print("  " * depth + f"{Path(d).name}/  ({len(files)} files)")
            if depth >= 3:
                dirs[:] = []
    raise FileNotFoundError("competition data not found: Add Input -> Competitions -> "
                            "shl-hiring-assessment-2026, then run this cell again")
print(f"competition data: {TRAIN_CSV}  |  audio: train {wav_dirs[AUDIO['train']]} wav, "
      f"test {wav_dirs[AUDIO['test']]} wav")

# Reuse caches from a previous run: attach an earlier version of this notebook as
# input (Add Input -> Your Work -> this notebook) and its caches are copied here,
# so unchanged stages resume instead of recomputing. Cache file names encode
# their settings, so a changed stage can never pick up a stale cache.
import shutil
CACHE_PATTERNS = ("asr_c.jsonl", "ctc.jsonl", "wavlm_layers.npz", "ssl_*.npz",
                  "deberta_*.npz", "ft_*.npz", "spk_xvec.npz")
restored, present = set(), set()
for pat in CACHE_PATTERNS:
    for src in sorted(walk_files(INPUT_ROOT, (pat,))):
        dst = OUT / src.name
        if dst.exists():
            present.add(src.name)
        else:
            shutil.copy2(src, dst)
            restored.add(src.name)
if restored:
    print("restored caches:", ", ".join(sorted(restored)))
elif present:
    print(f"caches already in place from earlier in this session ({len(present)} files)")
else:
    print("no caches attached — every stage computes from scratch")

train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
train.columns = [c.strip().lower() for c in train.columns]
test.columns  = [c.strip().lower() for c in test.columns]

# The 37 label-0.0 rows (IDs 5037-5073) are a degraded-audio sentinel, not a
# grammar score — see Findings. Dropped from the regressor.
train_all, train = train, train[train.label > 0].reset_index(drop=True)
print(f"train.csv {len(train_all)} rows -> {len(train)} after dropping "
      f"{len(train_all) - len(train)} zero-label | test.csv {len(test)}")

# Filenames collide across splits (train/audio_13.wav != test/audio_13.wav),
# so (split, filename) is the key everywhere.
jobs = ([("train", f) for f in train.filename] +
        [("test",  f) for f in test.filename])


def load16k(path):
    """Mono float32 at 16 kHz."""
    import soundfile as sf, librosa
    try:
        y_, sr_ = sf.read(str(path), dtype="float32", always_2d=False)
    except Exception:
        y_, sr_ = librosa.load(str(path), sr=None, mono=False)
        y_ = np.asarray(y_, dtype=np.float32)
    if y_.ndim > 1:
        y_ = y_.mean(axis=1 if y_.shape[0] > y_.shape[1] else 0)
    if sr_ != SR:
        y_ = librosa.resample(y_, orig_sr=sr_, target_sr=SR)
    return np.ascontiguousarray(y_, dtype=np.float32)


def load_done(p):
    """jsonl -> {(split, filename): record}. Skips a torn last line."""
    d, p = {}, Path(p)
    if p.exists():
        for line in p.read_text().splitlines():
            try:
                r = json.loads(line)
                d[(r["split"], r["filename"])] = r
            except json.JSONDecodeError:
                pass
    return d


print(f"cached transcripts: whisper {len(load_done(OUT / 'asr_c.jsonl'))}  "
      f"ctc {len(load_done(OUT / 'ctc.jsonl'))}")

In [ ]:
# v20 guard: this version only adds DeBERTa columns and depends on the attached caches
# (dataset amritarajput54/shl-grammar-caches-v18). Without them every stage recomputes (~7 h),
# and without CUDA it fails later anyway (v19 ran on a TPU machine). Stop here instead.
import torch
REQUIRED_CACHES = ["asr_c.jsonl", "ctc.jsonl", "wavlm_layers.npz", "ssl_wavlm_large.npz",
                   "ssl_whisper_med.npz", "ft_wavlm-base-plus_c8_e8.npz", "ft_wavlm-large_c8_e8.npz",
                   "spk_xvec.npz", "deberta_deberta-v3-base_whisper_e6.npz",
                   "deberta_deberta-v3-base_ctc_e6.npz", "deberta_deberta-v3-base_ctc_pair_e6.npz"]
missing = [f for f in REQUIRED_CACHES if not (OUT / f).exists()]
if missing:
    raise RuntimeError(f"caches not restored: {missing}")
if not torch.cuda.is_available():
    raise RuntimeError("no CUDA device: the accelerator must be a GPU")
print(f"guard ok: {len(REQUIRED_CACHES)} caches present, GPU {torch.cuda.get_device_name(0)}")


## 2 · ASR pass 1 — Whisper

Word timings, sentence boundaries, normalised transcript.

In [ ]:
# =============================================================================
# 2 · ASR pass 1 — Whisper small.en, resume-safe
#
#   - Audio goes to faster-whisper as a float32 array. Its own decoder calls
#     PyAV with a keyword PyAV 14 removed; ndarray input skips that path.
#   - Prompt-priming for disfluency recovery was tested and dropped: a
#     filler-heavy initial_prompt gave transcripts indistinguishable from
#     unprimed ones. Whisper deletes non-lexical fillers regardless, which is
#     why the CTC pass exists.
#   - fp16 here. int8 is the deployment configuration; quantising in this run
#     costs accuracy on the word timings the pause features depend on.
# =============================================================================

import json, time
from faster_whisper import WhisperModel

WHISPER = "small.en"
ASR_OUT = OUT / "asr_c.jsonl"

for split, d in AUDIO.items():
    n = len(list(d.glob("*.wav")))
    print(f"{split:<6} {n} wav")
    assert n > 0, f"no wav files under {d}"


def transcribe(model, audio):
    segs, info = model.transcribe(audio, language="en", beam_size=5, word_timestamps=True,
                                  condition_on_previous_text=False, vad_filter=False)
    S, Wd = [], []
    for s in segs:
        S.append(dict(start=round(s.start, 2), end=round(s.end, 2), text=s.text,
                      avg_logprob=round(s.avg_logprob, 4),
                      no_speech_prob=round(s.no_speech_prob, 4),
                      compression_ratio=round(s.compression_ratio, 3),
                      temperature=s.temperature))
        Wd += [[w.word, round(w.start, 2), round(w.end, 2), round(w.probability, 4)]
               for w in (s.words or [])]
    return dict(duration=round(info.duration, 2),
                text="".join(s["text"] for s in S).strip(), segments=S, words=Wd)


done = load_done(ASR_OUT)
todo = [j for j in jobs if j not in done]
print(f"{len(done)} cached, {len(todo)} to transcribe")

if todo:
    model = WhisperModel(WHISPER, device="cuda", compute_type="float16")
    t0, errors = time.time(), []
    for i, (split, f) in enumerate(todo, 1):
        try:
            r = dict(split=split, filename=f, **transcribe(model, load16k(AUDIO[split] / f)))
        except Exception as exc:
            errors.append((split, f, str(exc)))
            continue
        with open(ASR_OUT, "a") as fh:
            fh.write(json.dumps(r) + "\n")
        done[(split, f)] = r
        if i % 100 == 0 or i == len(todo):
            el = time.time() - t0
            print(f"  {i}/{len(todo)}  {el/60:.1f} min  ETA {el/i*(len(todo)-i)/60:.1f} min", flush=True)
    if errors:
        print(f"!! {len(errors)} failures:", errors[:10])
    del model

print(f"whisper transcripts: {len(done)}")

## 3 · ASR pass 2 — CTC

`wav2vec2-large-robust-ft-swbd-300h`, frozen. Pretrained on Libri-Light +
CommonVoice + Switchboard + Fisher, fine-tuned on conversational telephone
speech — spontaneous rather than read, and Switchboard transcribes filled
pauses explicitly.

Replaced `wav2vec2-base-960h`, which is LibriSpeech-only and collapsed content
words here ("playground" → "placland"). The swap moved `repair_wer` from −0.468
to −0.553 and cut the zero-filler share from 73% to 26%.

In [ ]:
# =============================================================================
# CTC pass v2 — wav2vec2-large-robust-ft-swbd-300h, frozen, greedy decode
#
# Why the swap from wav2vec2-base-960h:
#   base-960h is LibriSpeech only (read audiobooks, mostly US-native). On
#   spontaneous accented speech it collapsed content words
#   ("playground"->"placland", "merry-go-round"->"naivin rand") and produced
#   median repair_wer 0.365 with a p90 of 0.700. It also returned zero filled
#   pauses for 73% of files, killing filler_density.
#
#   robust-ft-swbd-300h is pretrained on Libri-Light + CommonVoice + Switchboard
#   + Fisher, fine-tuned on Switchboard conversational telephone speech. Two
#   reasons it fits better: spontaneous rather than read speech, and Switchboard
#   transcription conventions include filled pauses, so "uh"/"um" are tokens the
#   model was trained to emit.
#
# Baseline to beat (base-960h, 732 labelled train rows):
#   repair_wer_nofiller   raw -0.468   partial (vs conf) -0.242
#   pron_tail_depth       raw -0.396   partial (vs repair_wer) -0.146
#   filler_density        raw +0.041   73% of files zero
# =============================================================================

import json, time, warnings
import numpy as np
import torch
from pathlib import Path
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

warnings.filterwarnings("ignore")

CTC_MODEL = "facebook/wav2vec2-large-robust-ft-swbd-300h"
CHUNK_S   = 20.0          # large model, 1024-dim; 20s keeps attention in check
DEVICE_T  = "cuda" if torch.cuda.is_available() else "cpu"
CTC_OUT   = OUT / "ctc.jsonl"

# Expects from earlier cells: AUDIO, OUT, SR, jobs, load16k, load_done, train

ctc_done = load_done(CTC_OUT)
ctc_todo = [j for j in jobs if j not in ctc_done]
print(f"{len(ctc_done)} already done; {len(ctc_todo)} to go")


# ---- model ------------------------------------------------------------------
processor = Wav2Vec2Processor.from_pretrained(CTC_MODEL)
ctc_model = Wav2Vec2ForCTC.from_pretrained(CTC_MODEL).to(DEVICE_T).eval()

VOCAB    = {v: k for k, v in processor.tokenizer.get_vocab().items()}
BLANK_ID = ctc_model.config.pad_token_id
DELIM    = processor.tokenizer.word_delimiter_token
print(f"vocab {len(VOCAB)}  blank_id {BLANK_ID}  delim {DELIM!r}")


def ctc_words(audio, sr=SR):
    """-> list of [word, start, end, mean_log_posterior]."""
    out, step = [], int(sr * CHUNK_S)

    for i in range(0, len(audio), step):
        chunk = audio[i:i + step]
        if len(chunk) < sr * 0.1:
            continue
        offset = i / sr

        inputs = processor(chunk, sampling_rate=sr, return_tensors="pt", padding=True)
        with torch.inference_mode():
            logits = ctc_model(inputs.input_values.to(DEVICE_T)).logits[0]
        best = torch.log_softmax(logits.float(), dim=-1).max(dim=-1)
        ids  = best.indices.cpu().numpy()
        conf = best.values.cpu().numpy()
        frame_s = len(chunk) / sr / len(ids)

        chars, confs, start_f, prev = [], [], None, -1

        def flush(end_f):
            if chars and start_f is not None:
                out.append(["".join(chars).lower(),
                            round(offset + start_f * frame_s, 3),
                            round(offset + end_f * frame_s, 3),
                            round(float(np.mean(confs)), 4)])
            chars.clear(); confs.clear()

        for f, tid in enumerate(ids):
            tid = int(tid)
            if tid == BLANK_ID:
                prev = tid
                continue
            if tid == prev:                      # CTC repeat collapse
                confs.append(float(conf[f]))
                continue
            prev = tid
            tok = VOCAB.get(tid, "")
            if tok == DELIM:
                flush(f)
                start_f = None
                continue
            if start_f is None:
                start_f = f
            chars.append(tok)
            confs.append(float(conf[f]))
        flush(len(ids) - 1)

    return [w for w in out if w[0].strip()]


# ---- run --------------------------------------------------------------------
if ctc_todo:
    t0, errors = time.time(), []
    for i, (split, f) in enumerate(ctc_todo, 1):
        try:
            audio = load16k(AUDIO[split] / f)
            Wds = ctc_words(audio)
            r = dict(split=split, filename=f,
                     duration=round(len(audio) / SR, 2),
                     raw_text=" ".join(w[0] for w in Wds),
                     words=Wds)
        except Exception as exc:
            errors.append((split, f, str(exc)))
            continue
        with open(CTC_OUT, "a") as fh:
            fh.write(json.dumps(r) + "\n")
        ctc_done[(split, f)] = r

        if i % 100 == 0 or i == len(ctc_todo):
            el = time.time() - t0
            print(f"{i}/{len(ctc_todo)}  {el/60:.1f} min  "
                  f"ETA {el/i*(len(ctc_todo)-i)/60:.1f} min", flush=True)

    if errors:
        print(f"\n!! {len(errors)} failures:")
        for e in errors[:20]:
            print("  ", e)

print(f"\nctc done: {len(ctc_done)}")


# =============================================================================
# Before/after comparison — the only reason to have run this
# =============================================================================
import re
from scipy.stats import spearmanr

NON_LEXICAL = {"uh","uhh","uhhh","um","umm","ummm","erm","er","err","ah","ahh",
               "eh","ehh","mm","mmm","hmm","hm","huh","mhm","uhm"}
_W = re.compile(r"[a-z']+")
norm = lambda t: _W.findall((t or "").lower())


def lev(a, b):
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]


def partial(x, z, y):
    """spearman(x, y) with the linear part of z removed from x."""
    ok = np.isfinite(x) & np.isfinite(z) & np.isfinite(y)
    res = x[ok] - np.polyval(np.polyfit(z[ok], x[ok], 1), z[ok])
    return float(spearmanr(res, y[ok]).statistic)


W = load_done(OUT / "asr_c.jsonl")
rows = []
for k in sorted(set(W) & set(ctc_done)):
    w, c = W[k], ctc_done[k]
    wt, ct = norm(w["text"]), norm(c["raw_text"])
    wt_nf = [t for t in wt if t not in NON_LEXICAL]
    ct_nf = [t for t in ct if t not in NON_LEXICAL]
    cf = np.array([x[3] for x in c["words"]], dtype=float)
    rows.append({
        "split": k[0], "filename": k[1],
        "repair_wer_nofiller": lev(ct_nf, wt_nf) / max(len(ct_nf), 1),
        "pron_tail_depth": (float(np.median(cf) - np.percentile(cf, 5))
                            if cf.size >= 10 else np.nan),
        "ctc_conf_median": float(np.median(cf)) if cf.size else np.nan,
        "filler_density": sum(t in NON_LEXICAL for t in ct) / max(len(ct), 1),
        "filler_count": sum(t in NON_LEXICAL for t in ct),
        "n_words_ctc": len(ct),
    })

d2 = pd.DataFrame(rows)
t2 = d2[d2.split == "train"].merge(train, on="filename")
y  = t2.label.values
rw = t2.repair_wer_nofiller.values
td = t2.pron_tail_depth.values
cm = t2.ctc_conf_median.values

print("\n" + "=" * 70)
print(f"{'':<24}{'base-960h':>22}{'swbd-300h':>22}")
print(f"{'repair_wer raw':<24}{-0.468:>22.3f}{spearmanr(rw, y).statistic:>22.3f}")
print(f"{'repair_wer partial':<24}{-0.242:>22.3f}{partial(rw, cm, y):>22.3f}")
print(f"{'pron_tail_depth raw':<24}{-0.396:>22.3f}{spearmanr(td, y, nan_policy='omit').statistic:>22.3f}")
print(f"{'pron_tail_depth partial':<24}{-0.146:>22.3f}{partial(td, rw, y):>22.3f}")
print(f"{'filler_density raw':<24}{+0.041:>22.3f}{spearmanr(t2.filler_density, y).statistic:>22.3f}")
print(f"{'zero-filler share':<24}{0.73:>22.2f}{(d2.filler_count == 0).mean():>22.2f}")
print("=" * 70)
print("Read the raw rows and the sextile table. The partial rows are NOT")
print("comparable across models: the control (ctc_conf_median) changed its")
print("relationship to repair_wer when the model changed.")

print("\nlabel by repair_wer_nofiller sextile:")
print(t2.groupby(pd.qcut(t2.repair_wer_nofiller, 6, duplicates="drop"))
        .agg(n=("label", "size"), mean_label=("label", "mean")).round(2).to_string())

ex = ctc_done[("train", "audio_192.wav")]
print("\ntrain/audio_192.wav")
print("  CTC :", ex["raw_text"][:220])
print("  base: wel the the placland is very very beautiful ant thing therar there are flatins like the naivin rand the ling")

d2.to_csv(OUT / "ctc_diagnostic_swbd.csv", index=False)

## 4 · Features

21 columns, one row per file.

| group | measures | source |
|---|---|---|
| `midclause_pause_*`, `boundary_share`, `articulation_rate`, `phonation_ratio` | silence placement, speech rate | Whisper timings |
| `repetition_rate` | restarts and stutters | Whisper — it keeps structural disfluency even though it drops "um" |
| `text_ttr`, `mean_sent_len`, `subordinator_rate`, `modal_rate`, … | lexical range, syntactic complexity | Whisper text |
| `gpt2_nll_raw`, `gpt2_nll_gap` | how unusual the word sequences are | GPT-2 on the **raw** transcript |
| `repair_wer_nofiller` | how much Whisper had to rewrite the speaker | both |

**Dropped on measurement, not taste.** `filler_density` (−0.042 *with fillers
correctly detected* — the rubric mentions fillers nowhere); `pron_tail_depth`
(partial −0.074 against `repair_wer`); `ctc_conf_median` (sign flip under
residualisation); `pron_critical_rate` (constant zero — a median−k·MAD rule is
wrong for this skew); `sentence_rate` (flat across all six bins);
`asr_avg_logprob` (cost 0.002 RMSE, and "how confident was the recogniser" is
the accent exposure under another name).

`no_punct` is computed per Whisper **segment**, not by splitting the joined text
on ".". Segment-level gives −0.171, the naive version +0.036. 91 of 948 files
produce no sentence end at all — rubric level 2, "may leave sentences
incomplete".

In [ ]:
# =============================================================================
# Feature build — everything from cache. No GPU except the GPT-2 pass.
#
# Carried forward from this session's measurements:
#   KEEP   repair_wer_nofiller  spearman -0.552, monotonic across 6 bins,
#                               clipped at 2.0 (ratio explodes when CTC fails)
#   KEEP   no_punct             the cliff at zero; the gradient above it is flat
#   DROP   pron_tail_depth      partial vs repair_wer = -0.074
#   DROP   ctc_conf_median      sign flip under residualisation
#   DROP   pron_critical_rate   constant 0 — MAD rule wrong for this skew
#   DROP   filler_density       -0.042. Honest negative; the rubric never
#                               mentions fillers. Report it.
#   DROP   sentence_rate        bins flat 3.36/3.28/3.59/3.48/3.63/3.55
#
# Which pass supplies what, given the smoke test: Whisper deletes non-lexical
# fillers but PRESERVES structural disfluency ("there are, there are"). So
# repetition comes from Whisper; nothing needs the CTC text except repair_wer.
# =============================================================================

import re
import numpy as np
import pandas as pd

MIN_PAUSE_S      = 0.25
PAUSE_CAP_S      = 3.00
BOUNDARY_TOL_S   = 0.20
THOUGHT_GRACE    = 1          # forgive the longest mid-clause pause per file

SUBORDINATORS = {"because","although","though","while","whereas","since","unless",
                 "until","whether","if","when","after","before","that","which",
                 "who","whom","whose"}
CONNECTIVES   = {"however","therefore","moreover","furthermore","nevertheless",
                 "consequently","meanwhile","otherwise","instead","thus"}
MODALS        = {"would","could","should","might","must","may","shall","ought"}
NON_LEXICAL   = {"uh","uhh","um","umm","erm","er","ah","ahh","eh","mm","hmm","hm","uhm"}

_W = re.compile(r"[a-z']+")
norm = lambda t: _W.findall((t or "").lower())


def lev(a, b):
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]


# --------------------------------------------------------------------------- #
def pause_feats(words, bounds, dur):
    """Boundary-aware. A silence counts only if mid-clause — pausing after a
    completed sentence is prosody, not a grammar failure. Then the single
    longest remaining pause is forgiven: the cognitive-load allowance, as code
    rather than as a disclaimer."""
    out = dict(midclause_pause_ratio=0.0, midclause_pause_per_min=0.0,
               boundary_share=1.0, articulation_rate=0.0, phonation_ratio=0.0)
    if len(words) < 3 or dur <= 0:
        return out

    st = np.array([w[1] for w in words], float)
    en = np.array([w[2] for w in words], float)
    span = max(en[-1] - st[0], 1e-6)
    phon = float(np.clip(en - st, 0, None).sum())
    B = np.asarray(bounds, float)

    natural, mid = [], []
    for g, t in zip(st[1:] - en[:-1], en[:-1]):
        if g < MIN_PAUSE_S:
            continue
        at_b = B.size and np.min(np.abs(B - t)) <= BOUNDARY_TOL_S
        (natural if at_b else mid).append(min(float(g), PAUSE_CAP_S))

    mid.sort(reverse=True)
    counted = mid[THOUGHT_GRACE:]
    n_all = len(mid) + len(natural)

    out["midclause_pause_ratio"]   = sum(counted) / span
    out["midclause_pause_per_min"] = len(counted) / (span / 60.0)
    out["boundary_share"]          = len(natural) / n_all if n_all else 1.0
    out["articulation_rate"]       = len(words) / max(phon, 1e-6)
    out["phonation_ratio"]         = phon / span
    return out




def text_feats(clean, words):
    toks = norm(clean)
    content = [t for t in toks if t not in NON_LEXICAL]
    n = max(len(toks), 1)
    sents = [s for s in re.split(r"[.!?]+", clean or "") if s.strip()]

    # Structural disfluency — Whisper keeps these even though it drops "um".
    rep1 = sum(1 for a, b in zip(toks, toks[1:]) if a == b and len(a) > 1)
    rep2 = sum(1 for i in range(len(toks)-3) if toks[i:i+2] == toks[i+2:i+4])

    return {
        "n_words":            float(len(toks)),
        "text_ttr":           len(set(content)) / max(len(content), 1),
        "mean_sent_len":      len(toks) / max(len(sents), 1),   # guard: 91 files have 0
        "std_sent_len":       float(np.std([len(norm(s)) for s in sents])) if sents else 0.0,
        "subordinator_rate":  sum(t in SUBORDINATORS for t in toks) / n,
        "connective_rate":    sum(t in CONNECTIVES for t in toks) / n,
        "modal_rate":         sum(t in MODALS for t in toks) / n,
        "repetition_rate":    (rep1 + rep2) / n,
        "mean_word_len":      float(np.mean([len(t) for t in content])) if content else 0.0,
        "long_word_rate":     sum(len(t) >= 8 for t in content) / max(len(content), 1),
        
    }



# --------------------------------------------------------------------------- #
W = load_done(OUT / "asr_c.jsonl")
C = load_done(OUT / "ctc.jsonl")
keys = sorted(set(W) & set(C))
print(f"building features for {len(keys)} files")

rows = []
for k in keys:
    w, c = W[k], C[k]
    f = {"split": k[0], "filename": k[1], "duration": w["duration"]}

    bounds = [s["end"] for s in w["segments"] if s["text"].strip().endswith((".","!","?"))]
    f.update(pause_feats(w["words"], bounds, w["duration"]))
    f.update(text_feats(w["text"], w["words"]))

    wt_nf = [t for t in norm(w["text"])     if t not in NON_LEXICAL]
    ct_nf = [t for t in norm(c["raw_text"]) if t not in NON_LEXICAL]
    f["repair_wer_nofiller"] = min(lev(ct_nf, wt_nf) / max(len(ct_nf), 1), 2.0)

    f["asr_avg_logprob"] = float(np.mean([s["avg_logprob"] for s in w["segments"]])) \
                           if w["segments"] else 0.0
    rows.append(f)

 
F = pd.DataFrame(rows)

# Segment-level terminal punctuation. Splitting the joined text on "." gives
# +0.036; the segment-level test gives -0.171 — 91 files produce no sentence
# end at all, which maps onto rubric level 2 ("may leave sentences incomplete").
F["no_punct"] = [float(not any(s["text"].strip().endswith((".","!","?"))
                               for s in W[k]["segments"])) for k in keys]

# ---- GPT-2 perplexity on the RAW transcript --------------------------------
# Raw, because Whisper's text has already been regularised — its perplexity is
# flat across the score range. Ungrammatical sequences are low-probability under
# a model trained on well-formed English.
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained("gpt2")
    lm  = AutoModelForCausalLM.from_pretrained("gpt2").to(dev).eval()

    def nll(text):
        if not text or len(text.split()) < 3:
            return np.nan
        ids = tok(text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(dev)
        with torch.inference_mode():
            return float(lm(ids, labels=ids).loss)

    F["gpt2_nll_raw"]   = [nll(C[k]["raw_text"]) for k in keys]
    F["gpt2_nll_clean"] = [nll(W[k]["text"])     for k in keys]
    # The gap isolates ungrammaticality from topic difficulty, which lifts both.
    F["gpt2_nll_gap"]   = F.gpt2_nll_raw - F.gpt2_nll_clean
    print("gpt2 done")
except Exception as exc:
    print("gpt2 skipped:", exc)

F.to_csv(OUT / "features.csv", index=False)
print(f"feature matrix: {F.shape}")


# =============================================================================
# Ranking + CV
# =============================================================================
from scipy.stats import spearmanr
import lightgbm as lgb
from sklearn.model_selection import cross_val_predict, StratifiedKFold

T = F[F.split == "train"].merge(train, on="filename")
y = T.label.values
FEATS = [c for c in F.columns if c not in {"split", "filename", "label", "asr_avg_logprob"}]

print("\nspearman vs label:")
rank = (pd.Series({c: spearmanr(T[c], y, nan_policy="omit").statistic for c in FEATS})
          .sort_values(key=abs, ascending=False))
print(rank.round(3).to_string())

strat = T.label.replace({1.0: 1.5}).astype(str)
folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(T, strat))
base_rmse = np.sqrt(((y - y.mean()) ** 2).mean())


def cv(cols, label):
    X = T[cols].fillna(T[cols].median()).values
    oof = np.clip(cross_val_predict(
        lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=7,
                          min_child_samples=20, colsample_bytree=0.8,
                          reg_lambda=2.0, verbose=-1),
        X, y, cv=folds), 1.5, 5.0)
    r = np.sqrt(((y - oof) ** 2).mean())
    print(f"  {label:<34}{len(cols):>3} feat   RMSE {r:.4f}")
    return r, oof


print(f"\nbaseline (constant mean)              RMSE {base_rmse:.4f}")
cv(["repair_wer_nofiller"],                       "repair_wer only")
cv([c for c in FEATS if c.startswith(("mean_sent","std_sent","text_ttr","subordinator",
                                      "connective","modal","long_word","mean_word",
                                      "gpt2","no_punct"))], "text group only")
cv([c for c in FEATS if "pause" in c or c in {"articulation_rate","phonation_ratio",
                                              "boundary_share","repetition_rate"}],
   "fluency group only")
rmse_all, oof_all = cv(FEATS, "ALL")


print(f"\nhandcrafted features, LightGBM OOF RMSE {rmse_all:.4f}")

## 5 · Grammar-aware text features

Every handcrafted text feature is a *count* — type-token ratio, subordinator
rate, sentence length. None can see "he go to school yesterday". Two additions
ask the direct question:

- **`cola_*`** — RoBERTa fine-tuned on CoLA (the Corpus of Linguistic
  Acceptability) gives P(grammatical) for each sentence of the Whisper
  transcript. `cola_wmean` correlates +0.53 with the label — almost level with
  `repair_wer` — yet adds little to the model, because it overlaps with GPT-2
  and `repair_wer`. It stays because it is the explainable one: "38% of your
  sentences were judged ungrammatical" is something a candidate can act on, and
  the report cards in §9 use it.
- **`emb_pred`** — an `all-mpnet-base-v2` embedding of the transcript → ridge
  inside the CV folds → one out-of-fold column. Spearman +0.67; it took
  LightGBM from 0.69 to 0.63.

**Caveats.** `emb_pred` is a black box, and mpnet embeds meaning — so it may
partly learn *what* the speaker talked about rather than how grammatically.
That is separable only with prompt labels, which the data does not include.
Like every stacked column here it is built on the same folds used downstream:
standard out-of-fold stacking, mildly optimistic.

CoLA correlates −0.39 with `repair_wer`. A classifier that sees only normalised
text, with no acoustic input, agreeing moderately with `repair_wer` is partial
evidence that `repair_wer` measures grammar and not only recognition difficulty.

In [ ]:
# =============================================================================
# Grammar-aware features — the one thing the pipeline was missing.
#
# Every existing text feature is a COUNT (TTR, subordinator rate, sentence
# length). None can see "he go to school yesterday". The rubric is entirely
# about grammar, so ask the direct question:
#
#   cola_*      RoBERTa fine-tuned on CoLA (Corpus of Linguistic Acceptability)
#               outputs P(grammatical) per sentence. Aggregated per file.
#               Explainable: "38% of your sentences were judged ungrammatical."
#
#   emb_pred    Frozen sentence encoder -> ridge inside the CV folds -> ONE
#               out-of-fold column. The "heavy model compresses to one float"
#               pattern from the original plan, never built until now.
#
# Scored on the WHISPER transcript. Whisper repairs some grammar, so CoLA on its
# output measures the errors Whisper did NOT fix — complementary to repair_wer,
# which measures the ones it did. (CTC text is unusable here: "pletgound" would
# read as ungrammatical, confusing recognition error with grammar.)
#
# Needs from earlier cells: F, W, keys, train, folds, cv, OUT
# =============================================================================

import re, importlib.util, subprocess, sys
import numpy as np, pandas as pd, torch
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

dev = "cuda" if torch.cuda.is_available() else "cpu"


# ---- 1. CoLA acceptability --------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSequenceClassification
COLA = "textattack/roberta-base-CoLA"
ctok = AutoTokenizer.from_pretrained(COLA)
cmod = AutoModelForSequenceClassification.from_pretrained(COLA).to(dev).eval()


def p_all(sents, bs=64):
    out = []
    for i in range(0, len(sents), bs):
        b = ctok(sents[i:i+bs], return_tensors="pt", padding=True,
                 truncation=True, max_length=128).to(dev)
        with torch.inference_mode():
            out.append(torch.softmax(cmod(**b).logits.float(), -1).cpu().numpy())
    return np.vstack(out)


# Which output column means "grammatical"? Check rather than assume.
probe = p_all(["The cat sat on the mat.", "The cat sit on mat the."])
GOOD = int(np.argmax(probe[0] - probe[1]))
print(f"CoLA probe: good={probe[0, GOOD]:.3f}  bad={probe[1, GOOD]:.3f}  -> column {GOOD}")
assert probe[0, GOOD] > probe[1, GOOD] + 0.2, "CoLA probe did not separate — check the model"


def split_sents(text):
    s = [x.strip() for x in re.split(r"(?<=[.!?])\s+", text or "") if len(x.split()) >= 3]
    # 91 files have no terminal punctuation at all — score the whole text as one unit
    return s or ([text.strip()] if text and len(text.split()) >= 3 else [])


flat, owner = [], []
for k in keys:
    for s in split_sents(W[k]["text"]):
        flat.append(s); owner.append(k)
print(f"scoring {len(flat)} sentences across {len(keys)} files")
P = p_all(flat)[:, GOOD]

by = {}
for k, p, s in zip(owner, P, flat):
    by.setdefault(k, []).append((p, len(s.split())))

cola = {}
for k in keys:
    v = by.get(k, [])
    if not v:
        cola[k] = dict(cola_mean=np.nan, cola_wmean=np.nan, cola_min=np.nan, cola_frac_bad=np.nan)
        continue
    p = np.array([a for a, _ in v]); n = np.array([b for _, b in v], float)
    cola[k] = dict(cola_mean=p.mean(),
                   cola_wmean=(p * n).sum() / n.sum(),     # long sentences count more
                   cola_min=p.min(),
                   cola_frac_bad=(p < 0.5).mean())

kidx = list(zip(F.split, F.filename))
for c in ["cola_mean", "cola_wmean", "cola_min", "cola_frac_bad"]:
    F[c] = [cola.get(k, {}).get(c, np.nan) for k in kidx]


# ---- 2. sentence-embedding OOF ----------------------------------------------
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
from sentence_transformers import SentenceTransformer

st = SentenceTransformer("all-mpnet-base-v2", device=dev)
E = st.encode([W[k]["text"] for k in kidx], batch_size=32,
              normalize_embeddings=True, show_progress_bar=False)

tr_mask = (F.split == "train").values
te_mask = (F.split == "test").values
T_ = F[tr_mask].reset_index(drop=True).merge(train, on="filename", how="left")
assert len(T_) == tr_mask.sum() and T_.label.notna().all(), "train/label alignment broke"

Etr, Ete, ytr = E[tr_mask], E[te_mask], T_.label.values
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 30)))
oof, te = np.zeros(len(ytr)), np.zeros(te_mask.sum())
for a, b in folds:
    m = ridge.fit(Etr[a], ytr[a])
    oof[b] = m.predict(Etr[b]); te += m.predict(Ete) / len(folds)

F.loc[tr_mask, "emb_pred"] = oof
F.loc[te_mask, "emb_pred"] = te
print(f"emb_pred standalone OOF RMSE {np.sqrt(((ytr - np.clip(oof,1.5,5))**2).mean()):.4f}")


# ---- 3. does it help? -------------------------------------------------------
T = F[F.split == "train"].merge(train, on="filename")
y = T.label.values
NEW = ["cola_mean", "cola_wmean", "cola_min", "cola_frac_bad", "emb_pred"]

print("\nspearman vs label:")
for c in NEW:
    print(f"  {c:<16}{spearmanr(T[c], y, nan_policy='omit').statistic:+.3f}")

BASE_F = [c for c in F.columns
          if c not in {"split", "filename", "label", "asr_avg_logprob"} | set(NEW)]
med = T[BASE_F + NEW].median()

print()
_ = cv(BASE_F,                                   "current 21")
_ = cv(BASE_F + ["cola_mean","cola_wmean","cola_min","cola_frac_bad"], "+ CoLA")
_ = cv(BASE_F + ["emb_pred"],                    "+ emb_pred")
_ = cv(BASE_F + NEW,                             "+ both")

F.to_csv(OUT / "features.csv", index=False)
print("\nfeatures.csv updated. Rerun the modelling cell — it picks up the new")
print("columns automatically from F.columns and writes a new submission.csv.")

## 6 · Frozen speech encoders

Every feature before this one reaches the score through text; the audio entered only through hand-built pause and speech-rate features, which on their own barely beat a constant prediction.

In v3 a single frozen WavLM-base-plus — pooled over time, a ridge on top — was the strongest column in the notebook: 0.56 on its own, better than the fine-tuned DeBERTa reading the transcript (0.63), and it took the stacked LightGBM from 0.64 to 0.56. In v5 the frozen large encoders scored 0.53 on their own — WavLM-large 0.534, Whisper-medium 0.529, both with the best layers in the upper half — while fine-tuning WavLM-base-plus scored 0.57 at twelve minutes per fold. So this version adds two more large frozen encoders rather than more fine-tuning. It follows that evidence with encoders from other families, because different pretraining recipes encode different things:

| column | encoder | trained on |
|---|---|---|
| `wavlm_pred` | WavLM-base-plus | self-supervised, 94k h of English speech (v3's column) |
| `wavlm_large_pred` | WavLM-large | the same recipe, about 3× the parameters |
| `whisper_med_pred` | Whisper-medium, encoder only | supervised multilingual speech recognition, 680k h |
| `whisper_large_pred` | Whisper-large-v3, encoder only | the same task at larger scale; 32 layers |
| `hubert_large_pred` | HuBERT-large | self-supervised, a different masked-prediction recipe, 60k h |

Every encoder gets the same protocol: mean and standard deviation over time for every hidden layer, layers averaged uniformly, a ridge inside the same folds as everything else. No layer is chosen by validation score; the per-layer table only shows where the signal sits. A column must improve the stacked CV by at least 0.002 over everything kept before it, or the cell drops it and says so. Each encoder is wrapped, so a failed download skips that encoder instead of stopping the run.

**What these measure.** Pooling over time discards the order in which things were said, so these columns are weak instruments for sentence structure and strong ones for how the speaker *sounds* — pronunciation, rhythm, fluency, voice and recording conditions, accent included. That they predict a grammar rating better than a model reading the words is the most important result in the notebook; §10 discusses what it implies.

In [ ]:
# =============================================================================
# 6 · Speech SSL embeddings — several frozen encoders, one stacked column each
#
# In v3 one frozen WavLM-base-plus (pooled over time, ridge on top) was the
# strongest column in the notebook. Different pretraining recipes encode
# different things, so this version adds encoders from other families and lets
# each one earn its place:
#   wavlm        WavLM-base-plus    v3's column, unchanged (wavlm_pred)
#   wavlm_large  WavLM-large        same recipe, 3x the parameters
#   whisper_med  Whisper-medium     ENCODER only: supervised multilingual ASR
#   whisper_large, hubert_large     added in v5.1: in v4 the frozen large
#                encoders (0.53) beat fine-tuning (0.57) at a fraction of the cost
#
# Same honest protocol for every encoder: mean + std over time for every hidden
# layer, z-scored per layer on training rows, a UNIFORM average across layers,
# a ridge inside the seed-42 folds. No layer is chosen by validation score; the
# per-layer table is diagnostic only. A column is kept only if it improves the
# stacked LightGBM CV by >= KEEP_GAIN over everything kept before it. Each
# encoder is wrapped, so a download or memory failure skips that encoder
# instead of stopping the run.
# =============================================================================

import gc, math, time
import numpy as np, pandas as pd, torch, lightgbm as lgb
from transformers import AutoFeatureExtractor, AutoModel, Wav2Vec2FeatureExtractor, WhisperModel
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from scipy.stats import spearmanr

SSL_MODELS = {
    "wavlm":       "microsoft/wavlm-base-plus",
    "wavlm_large": "microsoft/wavlm-large",
    "whisper_med": "openai/whisper-medium",
    # "whisper_large": "openai/whisper-large-v3",    # v5.1: 32 layers, 128 mel bins (off in v20)
    # "hubert_large":  "facebook/hubert-large-ll60k",  # v5.1: a fourth pretraining recipe (off in v20)
}
SSL_CHUNK = 20.0                      # seconds per forward pass, wav2vec2-family encoders
KEEP_GAIN = 0.002
CLIP = (1.5, 5.0)
dev  = "cuda" if torch.cuda.is_available() else "cpu"
AMP  = dev == "cuda"
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float)) ** 2)))

kidx = list(zip(F.split, F.filename))

# Decoded audio is shared with §6b, so each file is read from disk at most once.
try:
    AUD
except NameError:
    AUD = {}


def wav(split, fn):
    k = (split, fn)
    if k not in AUD:
        AUD[k] = load16k(AUDIO[split] / fn)
    return AUD[k]


def cache_path(key):
    return OUT / ("wavlm_layers.npz" if key == "wavlm" else f"ssl_{key}.npz")


def _accumulate(hs, n_valid, acc):
    h = hs[:, :n_valid]                               # (layers, frames, dims)
    s, sq = h.sum(1), (h ** 2).sum(1)
    if acc is None:
        return [s, sq, n_valid]
    acc[0] += s; acc[1] += sq; acc[2] += n_valid
    return acc


def _finish(acc):
    S, SQ, n = acc
    mean = S / n
    std = (SQ / n - mean ** 2).clamp(min=0).sqrt()
    return torch.cat([mean, std], -1).cpu().numpy().astype(np.float32)


def pool_w2v(model, fe, audio):
    """wav2vec2-family: every hidden layer, mean + std over time."""
    acc, step = None, int(SR * SSL_CHUNK)
    for j in range(0, len(audio), step):
        chunk = audio[j:j + step]
        if len(chunk) < SR // 2:
            continue
        x = fe(chunk, sampling_rate=SR, return_tensors="pt").input_values.to(dev)
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            hs = model(x, output_hidden_states=True).hidden_states
        hs = torch.stack(hs)[:, 0].float()
        acc = _accumulate(hs, hs.shape[1], acc)
    return _finish(acc)


def pool_whisper(enc, fe, audio):
    """Whisper encoder: 30 s windows, padded frames excluded from the pooling
    (the encoder emits 50 frames per second of real audio)."""
    acc, step = None, 30 * SR
    for j in range(0, len(audio), step):
        chunk = audio[j:j + step]
        if len(chunk) < SR // 2:
            continue
        x = fe(chunk, sampling_rate=SR, return_tensors="pt").input_features.to(dev)
        n_valid = min(1500, math.ceil(len(chunk) / SR * 50))
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            hs = enc(x, output_hidden_states=True).hidden_states
        hs = torch.stack(hs)[:, 0].float()
        acc = _accumulate(hs, n_valid, acc)
    return _finish(acc)


def extract(key, name):
    p = cache_path(key)
    if p.exists():
        d = np.load(p, allow_pickle=True)
        if list(zip(d["split"], d["filename"])) == kidx:
            print(f"  {key}: features loaded from cache")
            return d["H"]
    if "whisper" in name:
        fe = AutoFeatureExtractor.from_pretrained(name)
        model, pool = WhisperModel.from_pretrained(name).encoder, pool_whisper
    else:
        try:
            fe = AutoFeatureExtractor.from_pretrained(name)
        except Exception:
            fe = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=SR, padding_value=0.0,
                                          do_normalize=True, return_attention_mask=True)
        model, pool = AutoModel.from_pretrained(name), pool_w2v
    model = model.to(dev).float().eval()               # fp32 weights, fp16 compute under autocast
    rows, t0 = [], time.time()
    for i, (split, fn) in enumerate(kidx, 1):
        rows.append(pool(model, fe, wav(split, fn)))
        if i % 200 == 0 or i == len(kidx):
            print(f"  {key}: {i}/{len(kidx)}  {(time.time() - t0) / 60:.1f} min", flush=True)
    H = np.stack(rows)
    np.savez(p, split=np.array([k[0] for k in kidx]), filename=np.array([k[1] for k in kidx]), H=H)
    del model
    gc.collect()
    if dev == "cuda":
        torch.cuda.empty_cache()
    return H


# ---- labels and folds, aligned to F's row order -----------------------------
tr_mask = (F.split == "train").values
te_mask = ~tr_mask
SPEECH = [f"{k}_pred" for k in SSL_MODELS]
F = F.drop(columns=[c for c in SPEECH if c in F.columns])      # idempotent on re-run
T = F[tr_mask].reset_index(drop=True).merge(train, on="filename", how="left")
assert T.label.notna().all(), "train/label alignment broke"
y = T.label.values.astype(float)
strat = T.label.replace({1.0: 1.5}).astype(str)
folds_ft = list(StratifiedKFold(5, shuffle=True, random_state=42).split(T, strat))


def ridge_oof(Xtr, Xte):
    m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(0, 6, 31)))
    oof, te = np.zeros(len(y)), np.zeros(len(Xte))
    for a, b in folds_ft:
        m.fit(Xtr[a], y[a])
        oof[b] = m.predict(Xtr[b])
        te += m.predict(Xte) / len(folds_ft)
    return oof, te


def cv_lgbm(frame, cols):
    X = frame[cols].fillna(frame[cols].median()).values
    o = cross_val_predict(lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=7,
                                            min_child_samples=20, colsample_bytree=0.8,
                                            reg_lambda=2.0, verbose=-1),
                          X, y, cv=folds_ft)
    return rmse(y, np.clip(o, *CLIP))


BASE_COLS = [c for c in F.columns if c not in {"split", "filename", "label", "asr_avg_logprob"}]
kept, report = [], []
for key, name in SSL_MODELS.items():
    col = f"{key}_pred"
    print(f"\n[{key}] {name}")
    try:
        H = extract(key, name)
    except Exception as exc:
        print(f"!! {key} skipped — {type(exc).__name__}: {str(exc)[:200]}")
        continue
    mu = H[tr_mask].mean(0, keepdims=True)
    sd = H[tr_mask].std(0, keepdims=True) + 1e-6
    Hz = (H - mu) / sd
    layer_r = [rmse(y, np.clip(ridge_oof(Hz[tr_mask, l], Hz[te_mask, l])[0], *CLIP))
               for l in range(Hz.shape[1])]
    print("  per-layer ridge OOF RMSE (diagnostic only):")
    print("  " + "  ".join(f"L{l}:{r:.3f}" for l, r in enumerate(layer_r)))

    X = Hz.mean(1)                                    # uniform average across layers
    oof, te = ridge_oof(X[tr_mask], X[te_mask])
    T[col] = oof
    r0, r1 = cv_lgbm(T, BASE_COLS + kept), cv_lgbm(T, BASE_COLS + kept + [col])
    keep = r1 <= r0 - KEEP_GAIN
    report.append((col, rmse(y, np.clip(oof, *CLIP)), spearmanr(oof, y).statistic, r0, r1, keep))
    print(f"  standalone OOF RMSE {report[-1][1]:.4f}   spearman {report[-1][2]:+.3f}")
    print(f"  stacked LightGBM: {r0:.4f} -> {r1:.4f}   ->  {'KEPT' if keep else 'DROPPED'}")
    if keep:
        kept.append(col)
        F.loc[tr_mask, col] = oof
        F.loc[te_mask, col] = te
    else:
        T = T.drop(columns=[col])

print("\nsummary")
for col, sa, sp, r0, r1, keep in report:
    print(f"  {col:<20} alone {sa:.4f}   stack {r0:.4f} -> {r1:.4f}   {'kept' if keep else 'dropped'}")
if len(kept) > 1:
    print("\nrank correlation between kept speech columns:")
    print(T[kept].corr(method="spearman").round(3).to_string())
F.to_csv(OUT / "features.csv", index=False)

## 6b · Fine-tuned speech models

§6 only fits a ridge on frozen, time-averaged features. Fine-tuning lets the encoder itself adapt to this rating — the usual recipe for automatic MOS prediction: a self-supervised speech encoder, mean pooling and a linear head, trained end to end.

Training sees random 8-second crops, two per clip per epoch: augmentation for 732 clips, and it keeps GPU memory flat. Each crop carries its clip's score. Prediction averages overlapping 8-second windows across the whole recording. The convolutional front end is frozen, layers are combined with learned weights, and the epoch count is fixed with no early stopping on the validation fold, so the out-of-fold score stays honest.

In v4 one seed of a fine-tuned WavLM-base-plus scored 0.567 on its own — level with the best frozen encoder. It costs about twelve minutes per fold on a T4, and the frozen large encoders in §6 do better at a fraction of that, so this version keeps the one cached seed. More seeds and a fine-tuned WavLM-large (smaller batches with gradient accumulation) are one line away in `FT_RUNS`. Each model becomes one column, gated like §6. Every (model, seed, fold) is cached, so v4's seed is reused and only new folds train. Each model is wrapped, so a failure such as running out of GPU memory skips that column only.

In [ ]:
# =============================================================================
# 6b · Fine-tuned speech models — several backbones and seeds, end to end
#
# The usual recipe for automatic MOS prediction: an SSL speech encoder, mean
# pooling and a linear head, trained end to end on the rating.
#
#   - training sees random CROP_S-second crops, CROPS_PER_CLIP per clip per
#     epoch: augmentation for 732 clips, flat GPU memory. Each crop carries its
#     clip's score.
#   - prediction averages overlapping windows (HOP_S hop) over the whole clip
#   - learned softmax weights over all hidden layers, mean pooling, linear head
#   - convolutional front end frozen, layerdrop off, target standardised per
#     fold, FIXED epoch count with no early stopping on the validation fold
#   - fp32 master weights under fp16 autocast; gradient accumulation lets the
#     large model train in smaller micro-batches
#   - same seed-42 folds as every stacked column; every (model, seed, fold) is
#     cached, so adding a seed trains only the new folds
#
# v4 trained WavLM-base-plus on one seed: 0.567 on its own, level with the best
# frozen encoder. v5 averages three seeds of it (seed averaging took DeBERTa
# from 0.644 to 0.634) and adds a fine-tuned WavLM-large. Each model becomes ONE
# stacked column, kept only if it improves the stacked CV. Each model is wrapped,
# so a failure (for example running out of GPU memory) skips that column only.
# =============================================================================

import gc, math, time, traceback
import numpy as np, pandas as pd, torch, torch.nn as nn, lightgbm as lgb
from transformers import (AutoConfig, AutoFeatureExtractor, AutoModel, Wav2Vec2FeatureExtractor,
                          get_cosine_schedule_with_warmup)
from sklearn.model_selection import StratifiedKFold, cross_val_predict

RUN_FT  = True
# v5.1: one seed of WavLM-base-plus, which v4 already cached, so this cell costs
# nothing. Fine-tuning takes ~12 min per fold on a T4 and scored 0.57 on its own,
# below the frozen large encoders in §6 (0.53) — so further seeds and WavLM-large
# are left switched off. Uncomment to add them; finished folds stay cached.
FT_RUNS = [
    # v20: as v18 ran it (every fold cached)
    dict(key="wavlm",       model="microsoft/wavlm-base-plus", seeds=(42, 7, 2024), bs=8, accum=1, lr=3e-5),
    dict(key="wavlm_large", model="microsoft/wavlm-large",   seeds=(42,),         bs=4, accum=2, lr=2e-5),
]
FT_EPOCHS      = 8
CROP_S, HOP_S  = 8.0, 4.0
CROPS_PER_CLIP = 2
EVAL_BS        = 16
LR_HEAD, WD, WARMUP = 1e-3, 0.01, 0.10
KEEP_GAIN      = 0.002
CLIP           = (1.5, 5.0)
dev            = "cuda" if torch.cuda.is_available() else "cpu"
AMP            = dev == "cuda"
CROP, HOP      = int(CROP_S * SR), int(HOP_S * SR)
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float)) ** 2)))
if dev == "cuda":
    torch.backends.cudnn.benchmark = True

try:
    AUD
except NameError:
    AUD = {}


def wav(split, fn):
    k = (split, fn)
    if k not in AUD:
        AUD[k] = load16k(AUDIO[split] / fn)
    return AUD[k]


def random_crop(w, rng):
    if len(w) <= CROP:
        return np.pad(w, (0, CROP - len(w)))
    s = int(rng.integers(0, len(w) - CROP + 1))
    return w[s:s + CROP]


def windows(w):
    if len(w) <= CROP:
        return [np.pad(w, (0, CROP - len(w)))]
    starts = list(range(0, len(w) - CROP + 1, HOP))
    if starts[-1] != len(w) - CROP:
        starts.append(len(w) - CROP)              # last window ends exactly at the clip end
    return [w[s:s + CROP] for s in starts]


def feature_extractor(name):
    try:
        return AutoFeatureExtractor.from_pretrained(name)
    except Exception:
        return Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=SR, padding_value=0.0,
                                        do_normalize=True, return_attention_mask=True)


class SSLReg(nn.Module):
    def __init__(self, name):
        super().__init__()
        cfg = AutoConfig.from_pretrained(name)
        cfg.layerdrop = 0.0
        self.bb = AutoModel.from_pretrained(name, config=cfg)
        fx = getattr(self.bb, "feature_extractor", None)
        if fx is not None:
            for p in fx.parameters():
                p.requires_grad = False
        self.lw = nn.Parameter(torch.zeros(cfg.num_hidden_layers + 1))
        self.head = nn.Linear(cfg.hidden_size, 1)

    def forward(self, x):
        hs = torch.stack(self.bb(x, output_hidden_states=True).hidden_states)   # (L, B, T, H)
        h = (torch.softmax(self.lw, 0)[:, None, None, None] * hs).sum(0)
        return self.head(h.mean(1)).squeeze(-1)


def predict(model, fe, clips):
    """Average over overlapping windows; one value per clip."""
    model.eval()
    owner, wins = [], []
    for i, w in enumerate(clips):
        ws = windows(w)
        wins += ws
        owner += [i] * len(ws)
    out = []
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
        for b in range(0, len(wins), EVAL_BS):
            x = fe(wins[b:b + EVAL_BS], sampling_rate=SR, return_tensors="pt").input_values.to(dev)
            out.append(model(x).float().cpu().numpy())
    out, owner = np.concatenate(out), np.array(owner)
    return np.array([out[owner == i].mean() for i in range(len(clips))])


def run_fold(run, fe, seed, tr_i, va_i):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    mu, sd = y[tr_i].mean(), y[tr_i].std()
    model = SSLReg(run["model"]).to(dev).float()
    bb_params = [p for p in model.bb.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([
        {"params": bb_params, "lr": run["lr"], "weight_decay": WD},
        {"params": [model.lw, *model.head.parameters()], "lr": LR_HEAD, "weight_decay": 0.0}])
    micro = math.ceil(len(tr_i) * CROPS_PER_CLIP / run["bs"])          # micro-batches per epoch
    total = FT_EPOCHS * math.ceil(micro / run["accum"])                 # optimiser steps, exact
    sch = get_cosine_schedule_with_warmup(opt, int(WARMUP * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    curve = []
    for ep in range(FT_EPOCHS):
        model.train()
        order = rng.permutation(np.repeat(tr_i, CROPS_PER_CLIP))
        batches = [order[i:i + run["bs"]] for i in range(0, len(order), run["bs"])]
        opt.zero_grad(set_to_none=True)
        for n, ids in enumerate(batches, 1):
            x = fe([random_crop(TR_W[i], rng) for i in ids], sampling_rate=SR,
                   return_tensors="pt").input_values.to(dev)
            yb = torch.tensor((y[ids] - mu) / sd, dtype=torch.float32, device=dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                pred = model(x)
            scaler.scale(nn.functional.mse_loss(pred.float(), yb) / run["accum"]).backward()
            if n % run["accum"] == 0 or n == len(batches):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sch.step()
                opt.zero_grad(set_to_none=True)
        if (ep + 1) % 2 == 0 or ep == FT_EPOCHS - 1:        # diagnostic only
            va = predict(model, fe, [TR_W[i] for i in va_i]) * sd + mu
            curve.append(f"ep{ep + 1}:{rmse(y[va_i], np.clip(va, *CLIP)):.3f}")

    va = predict(model, fe, [TR_W[i] for i in va_i]) * sd + mu    # final epoch, always
    te = predict(model, fe, TE_W) * sd + mu
    assert np.isfinite(va).all() and np.isfinite(te).all(), "non-finite predictions"
    del model, opt, sch, scaler
    gc.collect()
    if dev == "cuda":
        torch.cuda.empty_cache()
    return va, te, curve


def cv_lgbm(frame, cols):
    X = frame[cols].fillna(frame[cols].median()).values
    o = cross_val_predict(lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=7,
                                            min_child_samples=20, colsample_bytree=0.8,
                                            reg_lambda=2.0, verbose=-1),
                          X, y, cv=folds_ft)
    return rmse(y, np.clip(o, *CLIP))


FT_COLS = [f"ft_{r['key']}_pred" for r in FT_RUNS]
F = F.drop(columns=FT_COLS, errors="ignore")                    # idempotent on re-run
T  = F[F.split == "train"].merge(train, on="filename").reset_index(drop=True)
TE = F[F.split == "test"].reset_index(drop=True)
y  = T.label.values.astype(float)
strat = T.label.replace({1.0: 1.5}).astype(str)
folds_ft = list(StratifiedKFold(5, shuffle=True, random_state=42).split(T, strat))
TR_W = TE_W = None
kept_ft = []

for run in (FT_RUNS if RUN_FT else []):
    col = f"ft_{run['key']}_pred"
    cache_p = OUT / f"ft_{run['model'].split('/')[-1]}_c{int(CROP_S)}_e{FT_EPOCHS}.npz"
    print(f"\n[{col}] {run['model']}  seeds {run['seeds']}  batch {run['bs']}x{run['accum']}")
    try:
        cache = dict(np.load(cache_p)) if cache_p.exists() else {}
        need = any(f"s{s}_f{k}_va" not in cache for s in run["seeds"] for k in range(len(folds_ft)))
        fe = feature_extractor(run["model"])
        if need and TR_W is None:
            TR_W = [wav("train", f) for f in T.filename]
            TE_W = [wav("test", f) for f in TE.filename]
        oof, te_pred, t0 = np.zeros(len(y)), np.zeros(len(TE)), time.time()
        for seed in run["seeds"]:
            oof_s, te_s = np.zeros(len(y)), np.zeros(len(TE))
            for k, (a, b) in enumerate(folds_ft):
                kv, kt = f"s{seed}_f{k}_va", f"s{seed}_f{k}_te"
                if kv in cache:
                    va, tp, note = cache[kv], cache[kt], "cached"
                else:
                    va, tp, curve = run_fold(run, fe, seed, a, b)
                    cache[kv], cache[kt] = va, tp
                    np.savez(cache_p, **cache)
                    note = "val RMSE  " + " ".join(curve)
                print(f"  seed {seed:>4} fold {k}  {note}   [{(time.time() - t0) / 60:.1f} min]", flush=True)
                oof_s[b] = va
                te_s += tp / len(folds_ft)
            print(f"  seed {seed}: OOF RMSE {rmse(y, np.clip(oof_s, *CLIP)):.4f}")
            oof += oof_s / len(run["seeds"])
            te_pred += te_s / len(run["seeds"])
        print(f"  {col} standalone OOF RMSE ({len(run['seeds'])} seeds): {rmse(y, np.clip(oof, *CLIP)):.4f}")

        BASE = [c for c in F.columns if c not in {"split", "filename", "label", "asr_avg_logprob"}]
        T[col] = oof
        r0, r1 = cv_lgbm(T, BASE), cv_lgbm(T, BASE + [col])
        if r1 <= r0 - KEEP_GAIN:
            mp = {("train", f): v for f, v in zip(T.filename, oof)}
            mp.update({("test", f): v for f, v in zip(TE.filename, te_pred)})
            F[col] = [mp[(s, f)] for s, f in zip(F.split, F.filename)]
            kept_ft.append(col)
            print(f"  stacked LightGBM: {r0:.4f} -> {r1:.4f}   ->  KEPT")
        else:
            T = T.drop(columns=[col])
            print(f"  stacked LightGBM: {r0:.4f} -> {r1:.4f}   ->  DROPPED")
    except Exception as exc:
        print(f"!! {col} skipped — {type(exc).__name__}: {str(exc)[:300]}")
        traceback.print_exc(limit=3)
        F = F.drop(columns=[col], errors="ignore")
        T = T.drop(columns=[col], errors="ignore")
        gc.collect()
        if dev == "cuda":
            torch.cuda.empty_cache()

if not RUN_FT:
    print("RUN_FT is False — fine-tuned speech models skipped")
if len(kept_ft) > 1:
    print("\nrank correlation between kept fine-tuned columns:")
    print(T[kept_ft].corr(method="spearman").round(3).to_string())
F.to_csv(OUT / "features.csv", index=False)

## 6c · Speaker neighbours

Speaker-verification embeddings (x-vectors from `microsoft/wavlm-base-plus-sv`) place recordings of the same voice close together. This column predicts each clip's score from the scores of the training clips whose voices are nearest to it — a kernel-weighted average over the ten nearest — built out-of-fold like every other column: validation clips only see training-fold neighbours, test clips see all training clips. The number of neighbours and the kernel width are fixed in advance, not tuned on validation.

If the same people recorded several clips, this column picks that up directly, and the cell reports how many validation and test clips have a likely same-speaker match. If not, it is a weak "people who sound alike" predictor and the gate drops it.

This column recognises voices. That is legitimate within this competition's split, but it measures nothing about grammar; §10 and the write-up treat it accordingly.

In [ ]:
# =============================================================================
# 6c · Speaker neighbours — kNN on speaker embeddings
#
# x-vectors (microsoft/wavlm-base-plus-sv) place recordings of the same voice
# close together. This column predicts each clip's score from the scores of the
# training clips whose voices are closest to it: a kernel-weighted average over
# the K nearest, built out-of-fold like every other column — validation clips
# only see training-fold neighbours, test clips see all training clips.
#
# If the same people recorded several clips, this column picks that up directly;
# if not, it is a weak "people who sound alike" predictor and the gate drops it.
# K and the kernel width are fixed in advance, not tuned on validation.
# =============================================================================

import time, traceback
import numpy as np, pandas as pd, torch, lightgbm as lgb
from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector
from sklearn.model_selection import StratifiedKFold, cross_val_predict

SV_MODEL  = "microsoft/wavlm-base-plus-sv"
K_NN, TAU = 10, 0.05          # neighbours, kernel width on cosine similarity
SPK_THR   = 0.86              # model card's same-speaker threshold, for the diagnostic only
KEEP_GAIN = 0.002
CLIP      = (1.5, 5.0)
SPK_COL   = "spk_knn_pred"
dev = "cuda" if torch.cuda.is_available() else "cpu"
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float)) ** 2)))

try:
    AUD
except NameError:
    AUD = {}


def wav(split, fn):
    k = (split, fn)
    if k not in AUD:
        AUD[k] = load16k(AUDIO[split] / fn)
    return AUD[k]


def xvectors(keys):
    """Unit-length speaker embedding per clip: 20 s chunks, length-weighted mean."""
    fe = Wav2Vec2FeatureExtractor.from_pretrained(SV_MODEL)
    sv = WavLMForXVector.from_pretrained(SV_MODEL).to(dev).float().eval()
    rows, t0, step = [], time.time(), 20 * SR
    for i, (split, fn) in enumerate(keys, 1):
        audio, embs, wts = wav(split, fn), [], []
        for j in range(0, len(audio), step):
            chunk = audio[j:j + step]
            if len(chunk) < 3 * SR and embs:          # skip short tails once we have a chunk
                continue
            x = fe(chunk, sampling_rate=SR, return_tensors="pt").input_values.to(dev)
            with torch.inference_mode():
                e = torch.nn.functional.normalize(sv(input_values=x).embeddings, dim=-1)
            embs.append(e[0].float().cpu().numpy()); wts.append(len(chunk))
        v = np.average(embs, axis=0, weights=wts)
        rows.append(v / np.linalg.norm(v))
        if i % 200 == 0 or i == len(keys):
            print(f"  x-vectors {i}/{len(keys)}  {(time.time() - t0) / 60:.1f} min", flush=True)
    del sv
    if dev == "cuda":
        torch.cuda.empty_cache()
    return np.stack(rows).astype(np.float32)


def knn(Eq, Er, yr):
    S = Eq @ Er.T
    k = min(K_NN, Er.shape[0])
    idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    s = np.take_along_axis(S, idx, 1)
    w = np.exp((s - s.max(1, keepdims=True)) / TAU)
    return (w * yr[idx]).sum(1) / w.sum(1), S.max(1)


def cv_lgbm(frame, cols):
    X = frame[cols].fillna(frame[cols].median()).values
    o = cross_val_predict(lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=7,
                                            min_child_samples=20, colsample_bytree=0.8,
                                            reg_lambda=2.0, verbose=-1),
                          X, y, cv=folds_ft)
    return rmse(y, np.clip(o, *CLIP))


F = F.drop(columns=[SPK_COL], errors="ignore")
try:
    kidx = list(zip(F.split, F.filename))
    SPK_CACHE = OUT / "spk_xvec.npz"
    E = None
    if SPK_CACHE.exists():                            # any order: the leak check wrote one too
        d = np.load(SPK_CACHE, allow_pickle=True)
        m = {k: e for k, e in zip(zip(d["split"], d["filename"]), d["E"])}
        if all(k in m for k in kidx):
            E = np.stack([m[k] for k in kidx])
            print("speaker embeddings loaded from cache")
    if E is None:
        E = xvectors(kidx)
        np.savez(SPK_CACHE, split=np.array([k[0] for k in kidx]),
                 filename=np.array([k[1] for k in kidx]), E=E)

    tr_mask = (F.split == "train").values
    T = F[tr_mask].reset_index(drop=True).merge(train, on="filename", how="left")
    TE = F[~tr_mask].reset_index(drop=True)
    assert T.label.notna().all(), "train/label alignment broke"
    y = T.label.values.astype(float)
    strat = T.label.replace({1.0: 1.5}).astype(str)
    folds_ft = list(StratifiedKFold(5, shuffle=True, random_state=42).split(T, strat))

    Etr, Ete = E[tr_mask], E[~tr_mask]
    oof, best = np.zeros(len(y)), np.zeros(len(y))
    for a, b in folds_ft:
        oof[b], best[b] = knn(Etr[b], Etr[a], y[a])
    te, best_te = knn(Ete, Etr, y)

    same = best >= SPK_THR
    print(f"{SPK_COL} standalone OOF RMSE {rmse(y, np.clip(oof, *CLIP)):.4f}")
    print(f"  validation clips with a likely same-speaker training clip (cos >= {SPK_THR}): "
          f"{same.mean():.0%}" + (f"  — RMSE on them {rmse(y[same], np.clip(oof[same], *CLIP)):.3f}, "
                                  f"on the rest {rmse(y[~same], np.clip(oof[~same], *CLIP)):.3f}"
                                  if 0 < same.sum() < len(same) else ""))
    print(f"  test clips with a likely same-speaker training clip: {(best_te >= SPK_THR).mean():.0%}")

    BASE = [c for c in F.columns if c not in {"split", "filename", "label", "asr_avg_logprob"}]
    T[SPK_COL] = oof
    r0, r1 = cv_lgbm(T, BASE), cv_lgbm(T, BASE + [SPK_COL])
    if r1 <= r0 - KEEP_GAIN:
        mp = {("train", f): v for f, v in zip(T.filename, oof)}
        mp.update({("test", f): v for f, v in zip(TE.filename, te)})
        F[SPK_COL] = [mp[(s, f)] for s, f in zip(F.split, F.filename)]
        print(f"stacked LightGBM: {r0:.4f} -> {r1:.4f}   ->  {SPK_COL} KEPT")
    else:
        print(f"stacked LightGBM: {r0:.4f} -> {r1:.4f}   ->  {SPK_COL} DROPPED")
    F.to_csv(OUT / "features.csv", index=False)
except Exception as exc:
    print(f"!! speaker neighbours skipped — {type(exc).__name__}: {str(exc)[:300]}")
    traceback.print_exc(limit=3)
    F = F.drop(columns=[SPK_COL], errors="ignore")

## 7 · Fine-tuned transformer

DeBERTa-v3-base fine-tuned as a regressor on the Whisper transcripts, inside the same five folds. Mean pooling, dropout disabled (standard for transformer regression), target standardised per fold, and a **fixed** epoch count with no best-epoch selection on the validation fold, so the out-of-fold score stays honest.

The first run (4 epochs, 2 seeds) scored 0.644 on its own — better than all 21 handcrafted features combined — and took the 26-feature LightGBM stack from 0.633 to 0.612. In 7 of its 10 folds epoch 4 still beat epoch 3, so this version trains 6 epochs and averages 3 seeds. Six epochs made the seeds consistent — all three within 0.003 of each other, against 0.646 and 0.660 at four epochs — and averaging three of them took the column to 0.634.

With the audio embedding already in the stack, DeBERTa adds only about 0.01 (0.565 → 0.555). The two columns overlap heavily (r = 0.80).

In [ ]:
# =============================================================================
# 7 · Fine-tuned transformer — DeBERTa-v3-base regressor on the transcripts
#
# Design, chosen for honest CV on 732 rows:
#   - same seed-42 stratified folds as every other stacked column
#   - FIXED epoch count; no best-epoch selection on the validation fold
#   - mean pooling, dropout disabled (transformer-regression practice)
#   - target standardised per fold, inverted on prediction
#   - fp32 master weights (.float()) under fp16 autocast: recent transformers
#     load DeBERTa-v3 in its stored half precision, and the gradient scaler
#     cannot unscale fp16 gradients
#   - every (seed, fold) cached; the cache name carries model, input and epoch
#     count, so changing any of them can never silently reuse stale folds
#
# Output: one stacked column per run (RUNS below).
# =============================================================================

import gc, time
import numpy as np, pandas as pd, torch, torch.nn as nn, lightgbm as lgb
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from transformers import (AutoTokenizer, AutoModel, AutoConfig,
                          DataCollatorWithPadding, get_cosine_schedule_with_warmup)

MODEL_NAME = "microsoft/deberta-v3-base"
# v20: input -> column. Whisper repairs grammar before the "whisper" model sees it; the raw CTC
# text keeps the speaker's errors. "ctc_pair" = raw CTC [SEP] Whisper clean.
# v22: (model, input, column, lr). The base runs are cached; v3-large on the pair is new, at the usual
# large-model lr of 1e-5.
RUNS = [
    ("microsoft/deberta-v3-base",  "whisper",  "deberta_pred",            2e-5),
    ("microsoft/deberta-v3-base",  "ctc",      "deberta_ctc_pred",        2e-5),
    ("microsoft/deberta-v3-base",  "ctc_pair", "deberta_pair_pred",       2e-5),
    # v23 tried v3-large on the pair: 5/15 folds collapsed to the mean, alone 0.683, CV +0.0007 -> off
    # ("microsoft/deberta-v3-large", "ctc_pair", "deberta_large_pair_pred", 1e-5),
]
COLS = [r[2] for r in RUNS]
N_EPOCHS   = 6                  # at 4, 7 of 10 folds were still improving
LR, BS, WD, WARMUP = 2e-5, 8, 0.01, 0.10
SEEDS      = (42, 7, 2024)
GRAD_CKPT  = False              # switched on per model after an out-of-memory fold
USE_AMP    = True               # if predictions come back non-finite, set False
CLIP       = (1.5, 5.0)
dev        = "cuda" if torch.cuda.is_available() else "cpu"
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float)) ** 2)))

T  = F[F.split == "train"].merge(train, on="filename").reset_index(drop=True)
TE = F[F.split == "test"].reset_index(drop=True)
y  = T.label.values.astype(np.float64)
strat = T.label.replace({1.0: 1.5}).astype(str)
folds_ft = list(StratifiedKFold(5, shuffle=True, random_state=42).split(T, strat))


def text_for(split, fn):
    w, c = W[(split, fn)]["text"], C[(split, fn)]["raw_text"]
    return {"whisper": (w, None), "ctc": (c, None), "ctc_pair": (c, w), "pair": (w, c)}[INPUT]


try:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
except Exception:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

collate = DataCollatorWithPadding(tok)


class DS(Dataset):
    def __init__(self, pairs, yy=None):
        self.pairs, self.yy = pairs, yy

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        a, b = self.pairs[i]
        e = dict(tok(a, b, truncation=True, max_length=MAX_LEN))
        e.pop("token_type_ids", None)
        if self.yy is not None:
            e["labels"] = float(self.yy[i])
        return e


class Reg(nn.Module):
    def __init__(self):
        super().__init__()
        cfg = AutoConfig.from_pretrained(MODEL_NAME)
        cfg.hidden_dropout_prob = 0.0
        cfg.attention_probs_dropout_prob = 0.0
        self.bb = AutoModel.from_pretrained(MODEL_NAME, config=cfg)
        if GRAD_CKPT:
            self.bb.gradient_checkpointing_enable()
        self.head = nn.Linear(cfg.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        h = self.bb(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        return self.head((h * m).sum(1) / m.sum(1).clamp(min=1e-6)).squeeze(-1)


def predict(model, dl):
    model.eval()
    out = []
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        for b in dl:
            out.append(model(b["input_ids"].to(dev), b["attention_mask"].to(dev)).float().cpu().numpy())
    return np.concatenate(out)


def run_fold(seed, tr_i, va_i):
    torch.manual_seed(seed); np.random.seed(seed)
    mu, sd = y[tr_i].mean(), y[tr_i].std()
    yn = (y - mu) / sd
    g = torch.Generator(); g.manual_seed(seed)
    tr_dl = DataLoader(DS([tr_txt[i] for i in tr_i], yn[tr_i]), batch_size=BS,
                       shuffle=True, collate_fn=collate, generator=g)
    va_dl = DataLoader(DS([tr_txt[i] for i in va_i]), batch_size=32, collate_fn=collate)
    te_dl = DataLoader(DS(te_txt), batch_size=32, collate_fn=collate)

    model = Reg().to(dev).float()
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
    opt = torch.optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
         "weight_decay": WD},
        {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)],
         "weight_decay": 0.0}], lr=LR)
    steps = N_EPOCHS * len(tr_dl)
    sch = get_cosine_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    curve = []
    for ep in range(N_EPOCHS):
        model.train()
        for b in tr_dl:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                pred = model(b["input_ids"].to(dev), b["attention_mask"].to(dev))
            loss = nn.functional.mse_loss(pred.float(), b["labels"].float().to(dev))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        curve.append(rmse(y[va_i], np.clip(predict(model, va_dl) * sd + mu, *CLIP)))

    va = predict(model, va_dl) * sd + mu        # final epoch, always
    te = predict(model, te_dl) * sd + mu
    assert np.isfinite(va).all() and np.isfinite(te).all(), \
        "non-finite predictions — set USE_AMP = False and rerun this cell"
    del model, opt, sch, scaler
    gc.collect(); torch.cuda.empty_cache()
    return va, te, curve


for MODEL_NAME, INPUT, COL, LR in RUNS:
    try:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    except Exception:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    collate = DataCollatorWithPadding(tok)
    GRAD_CKPT = False
    CACHE = OUT / f"deberta_{MODEL_NAME.split('/')[-1]}_{INPUT}_e{N_EPOCHS}.npz"
    tr_txt = [text_for("train", f) for f in T.filename]
    te_txt = [text_for("test",  f) for f in TE.filename]
    lens = [len(tok(a, b)["input_ids"]) for a, b in tr_txt]
    MAX_LEN = int(min(512, np.percentile(lens, 99) + 16))
    print(f"\n[{MODEL_NAME.split('/')[-1]} | {INPUT}] -> {COL}   lr {LR}   tokens/transcript: median {int(np.median(lens))}  "
          f"p99 {int(np.percentile(lens, 99))}  -> MAX_LEN {MAX_LEN}  "
          f"({sum(l > MAX_LEN for l in lens)} truncated)")
    cache = dict(np.load(CACHE)) if CACHE.exists() else {}
    oof, te_pred = np.zeros(len(y)), np.zeros(len(TE))
    t0 = time.time()
    for seed in SEEDS:
        oof_s, te_s = np.zeros(len(y)), np.zeros(len(TE))
        for k, (a, b) in enumerate(folds_ft):
            kv, kt = f"s{seed}_f{k}_va", f"s{seed}_f{k}_te"
            if kv in cache:
                va, tp, note = cache[kv], cache[kt], "cached"
            else:
                oom = False
                try:
                    va, tp, curve = run_fold(seed, a, b)
                except torch.cuda.OutOfMemoryError:
                    oom = True
                if oom:                       # retry outside the except, so the failed fold's memory is freed
                    gc.collect(); torch.cuda.empty_cache()
                    GRAD_CKPT = True
                    print("  out of memory -> gradient checkpointing on for this model", flush=True)
                    va, tp, curve = run_fold(seed, a, b)
                cache[kv], cache[kt] = va, tp
                np.savez(CACHE, **cache)
                note = "val RMSE by epoch  " + " ".join(f"{c:.3f}" for c in curve)
            print(f"  seed {seed:>4} fold {k}  {note}   [{(time.time()-t0)/60:.1f} min]", flush=True)
            oof_s[b] = va
            te_s += tp / len(folds_ft)
        print(f"  seed {seed}: OOF RMSE {rmse(y, np.clip(oof_s, *CLIP)):.4f}")
        oof += oof_s / len(SEEDS)
        te_pred += te_s / len(SEEDS)

    print(f"{COL} standalone OOF RMSE ({len(SEEDS)} seeds): {rmse(y, np.clip(oof, *CLIP)):.4f}")

    mp = {("train", f): v for f, v in zip(T.filename, oof)}
    mp.update({("test", f): v for f, v in zip(TE.filename, te_pred)})
    F[COL] = [mp[(s, f)] for s, f in zip(F.split, F.filename)]
T = F[F.split == "train"].merge(train, on="filename").reset_index(drop=True)


def cv_lgbm(cols):
    X = T[cols].fillna(T[cols].median()).values
    o = cross_val_predict(lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=7,
                                            min_child_samples=20, colsample_bytree=0.8,
                                            reg_lambda=2.0, verbose=-1),
                          X, y, cv=folds_ft)
    return rmse(y, np.clip(o, *CLIP))


BASE_COLS = [c for c in F.columns
             if c not in {"split", "filename", "label", "asr_avg_logprob", *COLS}]
print(f"stacked LightGBM, same folds:   without DeBERTa {cv_lgbm(BASE_COLS):.4f}")
for col in COLS:
    print(f"  + {col:<18} {cv_lgbm(BASE_COLS + [col]):.4f}")
print(f"  + all               {cv_lgbm(BASE_COLS + COLS):.4f}")
for col in COLS:
    for c in ("emb_pred", "wavlm_pred", "deberta_pred"):
        if c in T.columns and c != col:
            print(f"  r({col}, {c}) = {T[col].corr(T[c], method='spearman'):+.3f}")
F.to_csv(OUT / "features.csv", index=False)

## 8 · Data checks and adversarial validation

**Duplicates.** Two checks the original plan called for. Near-identical transcripts within the training set would mean repeated speakers or prompts, and an optimistic CV; a test transcript matching a training one would be a leak. Because filenames collide across splits (§10), same-named pairs are compared directly as well.

**Recording length.** Test recordings average 48.8 s against 55.7 s in train. The duration quantiles show whether that looks like a cap or simply a different population.

**Adversarial validation.** Can a classifier tell train rows from test rows? It can: AUC ≈0.75, and the shift traces to recording length (KS 0.45, nearly triple the next feature). `text_ttr` follows mechanically, because type-token ratio falls as a transcript lengthens. In v3 four responses were measured: dropping `duration` lowered the AUC only to ≈0.69 and cost 0.035 RMSE; replacing it with words-per-second cost 0.026–0.078; `repair_wer` is length-independent (r = 0.14 with duration). `duration` stays, and §9's `DROP_DURATION` switch tests the question CV cannot answer.

The learned `*_pred` columns are excluded from this check: their training values are out-of-fold predictions and their test values are fold averages, so a classifier could separate them for purely mechanical reasons.

In [ ]:
# =============================================================================
# 8 · Data checks and adversarial validation — playbook technique 1
#
# (a) Duplicates. Near-identical transcripts within train mean repeated speakers
#     or prompts, which would make CV optimistic. A test transcript that matches
#     a train transcript would be a leak. Same-named files are compared too.
# (b) Recording length. If the test set was capped or trimmed, its duration
#     distribution shows it.
# (c) Adversarial validation: can a classifier tell train rows from test rows?
#     AUC ~0.5 means the splits are exchangeable and CV is a fair estimate.
#     Learned *_pred columns are excluded: train values are out-of-fold, test
#     values are fold averages, so they separate for mechanical reasons.
# =============================================================================

import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

# ---- (a) duplicates ---------------------------------------------------------
DUP = 0.6                                  # bigram/trigram TF-IDF cosine
trk = [("train", f) for f in train.filename]
tek = [("test", f) for f in test.filename]
V = TfidfVectorizer(ngram_range=(2, 3), sublinear_tf=True).fit_transform(
        [W[k]["text"] for k in trk + tek])
S_tr = cosine_similarity(V[:len(trk)])
np.fill_diagonal(S_tr, 0)
S_te = cosine_similarity(V[len(trk):], V[:len(trk)])
best = S_te.max(1)
row_of = {f: i for i, (_, f) in enumerate(trk)}
same = [(i, row_of[f]) for i, (_, f) in enumerate(tek) if f in row_of]
sim_same = np.array([S_te[i, j] for i, j in same])

print("(a) near-duplicate transcripts, bigram/trigram cosine >= "
      f"{DUP}:")
print(f"    within train        {int((np.triu(S_tr, 1) >= DUP).sum())} pairs")
print(f"    test -> any train   {int((best >= DUP).sum())} of {len(tek)} test files "
      f"(median best match {np.median(best):.2f})")
print(f"    same-named pairs    median similarity {np.median(sim_same):.2f} over {len(same)} pairs "
      "(1.0 would mean the same recording)")
if (best >= DUP).any():
    for i in np.argsort(-best)[:5]:
        if best[i] >= DUP:
            print(f"      {tek[i][1]} ~ train/{trk[S_te[i].argmax()][1]}  similarity {best[i]:.2f}")

# ---- (b) recording length ---------------------------------------------------
d_tr = F.loc[F.split == "train", "duration"]
d_te = F.loc[F.split == "test", "duration"]
q = [0.05, 0.25, 0.5, 0.75, 0.95]
print("\n(b) duration quantiles (5/25/50/75/95%)")
print(f"    train {np.round(d_tr.quantile(q).values, 1)}")
print(f"    test  {np.round(d_te.quantile(q).values, 1)}")
print(f"    test files within 1 s of the test maximum: {(d_te >= d_te.max() - 1).mean():.0%} "
      "(a large share suggests a cap)")

# ---- (c) adversarial validation ---------------------------------------------
AV  = [c for c in F.columns
       if c not in {"split", "filename", "label", "asr_avg_logprob"}
       and not c.endswith("_pred") and not c.startswith("cola_")]
lab = (F.split == "test").astype(int).values


def adv_auc(cols):
    A = F[cols].fillna(F[cols].median())
    p = cross_val_predict(lgb.LGBMClassifier(n_estimators=300, num_leaves=15, verbose=-1),
                          A, lab, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                          method="predict_proba")[:, 1]
    return roc_auc_score(lab, p)


print(f"\n(c) adversarial AUC, handcrafted features : {adv_auc(AV):.4f}")
print(f"    adversarial AUC, without duration     : {adv_auc([c for c in AV if c != 'duration']):.4f}")

tr_, te_ = F[F.split == "train"], F[F.split == "test"]
ks = sorted(((ks_2samp(tr_[c].dropna(), te_[c].dropna()).statistic, c) for c in AV), reverse=True)
print("\n    largest train/test shifts (KS):")
for s, c in ks[:5]:
    print(f"      {s:.3f}  {c:<24} train {tr_[c].mean():8.3f}   test {te_[c].mean():8.3f}")

## 9 · Modelling, interpretation, report cards

Stratified 5-fold on the exact label value (1.0 merged into 1.5 — a single sample), 9 model families × 3 seeds, hill-climbed blend, clipped to [1.5, 5.0]: only 4 training samples sit below 2.0.

**Seed-robust blend.** Each family is averaged across its three seeds *before* blending; hill-climbing over individual (family, seed) columns rewards whichever fold split happened to be lucky, so that per-seed blend is printed only for reference. Every learned column (`*_pred`) also enters the blend directly.

**`DROP_DURATION`** (off by default) removes recording length from the final model. Test recordings are about 7 s shorter than training ones and short recordings are scored lower; CV cannot tell whether that is fair to the test speakers, so this switch settles it with one submission.

**Reading the plots.**
- *Residuals* fall on diagonal stripes because labels take nine discrete values: each stripe is one true score. The vertical edge at 5.0 is the clip.
- *Predicted vs true* pulls both ends toward the middle. That is what a calibrated predictor does when its correlation with the label is below 1, not a defect: in v3 the blend's RMSE already equalled the best any linear rescaling could reach, σ·√(1 − r²), so stretching the predictions cannot help.
- *MAE by true score* shows where not to trust the model: error is largest at the ends of the scale, and the four speakers below 2.0 are effectively unlearnable.

**What the SHAP plots explain.** They — and the required training RMSE — come from a LightGBM model refit on all training rows. LightGBM can carry little or no weight in the final blend, so it is a *surrogate*: it shows which features carry signal for models of this kind, not the blend's exact arithmetic. The grouped chart sums SHAP within each feature family, so correlated columns don't dilute each other, and prints how much of the score the learned black-box columns carry.

**Report cards** describe individual test recordings using explainable features only, in terms a candidate could act on. The score on each card comes from the full model, so its lines cover only the explainable part of what drives that score. The transcriber line inherits `repair_wer`'s accent caveat (§10).

In [ ]:
# =============================================================================
# 9 · Modelling — ablation, diverse baselines, seed-robust blend,
#     interpretation, report cards, submission
# =============================================================================

import numpy as np, pandas as pd, lightgbm as lgb, xgboost as xgb, shap
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import RidgeCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import StratifiedKFold
from scipy.stats import pearsonr, spearmanr

CLIP = (1.5, 5.0)          # only 4 training samples sit below 2.0
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float)) ** 2)))
DROP_DURATION = False     # True removes recording length from the final model — see §8 (b)

EXCLUDE = {"split", "filename", "label", "asr_avg_logprob"} | ({"duration"} if DROP_DURATION else set())
FEATS = [c for c in F.columns if c not in EXCLUDE]
CORE  = ["repair_wer_nofiller", "gpt2_nll_raw", "gpt2_nll_gap", "n_words", "text_ttr",
         "articulation_rate", "mean_word_len", "mean_sent_len", "repetition_rate", "no_punct"]

T  = F[F.split == "train"].merge(train, on="filename").reset_index(drop=True)
TE = F[F.split == "test"].reset_index(drop=True)
y  = T.label.values.astype(float)
med = T[FEATS].median()
strat = T.label.replace({1.0: 1.5}).astype(str)
SEEDS_CV = (42, 1337, 2024)
FOLDS = {s: list(StratifiedKFold(5, shuffle=True, random_state=s).split(T, strat))
         for s in SEEDS_CV}
print(f"train {len(T)}  test {len(TE)}  features {len(FEATS)}")


def oof_test(model, cols, folds):
    """-> (oof, test_pred), both clipped. Test prediction is the fold average."""
    X, Xt = T[cols].fillna(med[cols]).values, TE[cols].fillna(med[cols]).values
    oof, te = np.zeros(len(y)), np.zeros(len(Xt))
    for tr_i, va_i in folds:
        m = clone(model).fit(X[tr_i], y[tr_i])
        oof[va_i] = m.predict(X[va_i])
        te += m.predict(Xt) / len(folds)
    return np.clip(oof, *CLIP), np.clip(te, *CLIP)


def hill_climb(O, yy, iters=200, steps=(.5, .35, .2, .1, .05, .02)):
    """Greedy convex blend (playbook technique 4); candidates scored in one array op."""
    single = np.sqrt(((O - yy[:, None]) ** 2).mean(0))
    j = int(single.argmin())
    cur, best = O[:, j].copy(), float(single[j])
    w = np.zeros(O.shape[1]); w[j] = 1.0
    for _ in range(iters):
        moved = False
        for a in steps:
            cand = (1 - a) * cur[:, None] + a * O
            sc = np.sqrt(((cand - yy[:, None]) ** 2).mean(0))
            k = int(sc.argmin())
            if sc[k] < best - 1e-7:
                cur, best = cand[:, k], float(sc[k])
                w = (1 - a) * w; w[k] += a; moved = True
                break
        if not moved:
            break
    return w, best


# ---- 1. ablation: does the explainable core match the full set? -------------
lgbm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=7,
                         min_child_samples=20, colsample_bytree=0.8,
                         reg_lambda=2.0, verbose=-1)
r_all, _  = oof_test(lgbm, FEATS, FOLDS[42])
r_core, _ = oof_test(lgbm, CORE,  FOLDS[42])
gap = rmse(y, r_core) - rmse(y, r_all)
USE = CORE if gap <= 0.02 else FEATS
print(f"\nablation:  all {len(FEATS)} -> {rmse(y, r_all):.4f}   core {len(CORE)} -> "
      f"{rmse(y, r_core):.4f}   gap {gap:+.4f}  ->  using {'CORE' if USE is CORE else 'ALL'}")


# ---- 2. diverse baselines × 3 seeds (playbook techniques 2 and 7) -----------
def zoo(seed):
    return {
        "ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30))),
        "krr":   make_pipeline(StandardScaler(), KernelRidge(alpha=1.0, kernel="rbf", gamma=0.05)),
        "svr":   make_pipeline(StandardScaler(), SVR(C=3.0, epsilon=0.1)),
        "knn":   make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=25, weights="distance")),
        "rf":    RandomForestRegressor(n_estimators=500, min_samples_leaf=4,
                                       max_features=0.6, random_state=seed, n_jobs=-1),
        "et":    ExtraTreesRegressor(n_estimators=500, min_samples_leaf=4,
                                     max_features=0.7, random_state=seed, n_jobs=-1),
        "mlp":   make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32), alpha=1e-2, max_iter=1500,
                                                          random_state=seed, early_stopping=True)),
        "lgbm":  lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=7,
                                   min_child_samples=20, colsample_bytree=0.8,
                                   reg_lambda=2.0, random_state=seed, verbose=-1),
        "xgb":   xgb.XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=3,
                                  min_child_weight=8, subsample=0.8, colsample_bytree=0.7,
                                  reg_lambda=2.0, random_state=seed, verbosity=0),
    }


families = list(zoo(0))
OOF  = {f: [] for f in families}
TEST = {f: [] for f in families}
for s in SEEDS_CV:
    for f, m in zoo(s).items():
        o, t = oof_test(m, USE, FOLDS[s])
        OOF[f].append(o); TEST[f].append(t)

# Each family is averaged across its seeds BEFORE blending: hill-climbing over
# individual (family, seed) columns rewards whichever fold split got lucky.
# The learned columns are standalone predictors too, so they enter directly.
direct = [c for c in T.columns if c.endswith("_pred")]
cand = families + direct
CO = np.column_stack([np.mean(OOF[f], 0) for f in families] +
                     [np.clip(T[c].values, *CLIP) for c in direct])
CT = np.column_stack([np.mean(TEST[f], 0) for f in families] +
                     [np.clip(TE[c].values, *CLIP) for c in direct])

board = pd.DataFrame({"candidate": cand, "oof_rmse": [rmse(y, CO[:, i]) for i in range(len(cand))]})
print("\ncandidates (families averaged over 3 seeds; learned columns direct):")
print(board.sort_values("oof_rmse").to_string(index=False, float_format="%.4f"))


# ---- 3. blend -----------------------------------------------------------------
w, _ = hill_climb(CO, y)
blend_oof  = np.clip(CO @ w, *CLIP)
blend_test = np.clip(CT @ w, *CLIP)
print(f"\nseed-robust blend   OOF RMSE {rmse(y, blend_oof):.4f}")
for wt, nm in sorted(((wt, cand[i]) for i, wt in enumerate(w) if wt > 1e-3), reverse=True):
    print(f"    {wt:.3f}  {nm}")

PS = np.column_stack([o for f in families for o in OOF[f]])
w_ps, _ = hill_climb(PS, y)
print(f"  reference only — blend over the 27 per-seed columns: "
      f"{rmse(y, np.clip(PS @ w_ps, *CLIP)):.4f}  (not used: rewards lucky fold splits)")


# ---- 4. metrics -------------------------------------------------------------
Xdf = T[USE].fillna(med[USE])
fit_all = clone(lgbm).fit(Xdf.values, y)
train_rmse = rmse(y, np.clip(fit_all.predict(Xdf.values), *CLIP))

print("\n" + "=" * 64)
print(f"  baseline (constant mean)        {rmse(y, np.full(len(y), y.mean())):.4f}")
print(f"  TRAINING RMSE  (required)       {train_rmse:.4f}   LightGBM refit on all rows")
print(f"  OOF CV RMSE    (honest)         {rmse(y, blend_oof):.4f}")
print(f"  generalisation gap              {rmse(y, blend_oof) - train_rmse:+.4f}")
print(f"  MAE                             {np.abs(y - blend_oof).mean():.4f}")
print(f"  Pearson / Spearman              {pearsonr(y, blend_oof)[0]:.3f} / "
      f"{spearmanr(y, blend_oof).statistic:.3f}")
print(f"  within 0.5 MOS                  {np.mean(np.abs(y - blend_oof) <= 0.5):.1%}")
print("=" * 64)
print("  Training RMSE on 732 rows measures memorisation. The OOF number is the\n"
      "  result; the gap is why.")


# ---- 5. interpretation --------------------------------------------------------
# NB: no matplotlib.use("Agg") — that backend switches off inline display, which
# is why the previous version's plots never rendered.
sv = shap.TreeExplainer(fit_all).shap_values(Xdf)
shap.summary_plot(sv, Xdf, show=False, max_display=15)
plt.title("Per-feature SHAP — LightGBM refit on all training rows")
plt.tight_layout(); plt.savefig("shap_summary.png", dpi=130, bbox_inches="tight"); plt.show()

# Grouped attribution: SHAP summed within each family before taking |.|, so
# correlated columns in one family do not dilute each other.
GROUPS = {
    "fine-tuned transformer (DeBERTa)": [c for c in USE if c.startswith("deberta_")],
    "speech models (frozen, fine-tuned, speaker kNN)": [c for c in USE if c.endswith("_pred")
                                            and not c.startswith("deberta_") and c != "emb_pred"],
    "sentence embedding (mpnet)":       ["emb_pred"],
    "grammaticality classifier (CoLA)": [c for c in USE if c.startswith("cola_")],
    "ASR repair (Whisper vs CTC)":      ["repair_wer_nofiller"],
    "language-model fluency (GPT-2)":   [c for c in USE if c.startswith("gpt2_")],
    "lexical & syntactic counts":       ["n_words", "text_ttr", "mean_sent_len", "std_sent_len",
                                         "subordinator_rate", "connective_rate", "modal_rate",
                                         "mean_word_len", "long_word_rate", "no_punct"],
    "pausing & speech rate":            ["midclause_pause_ratio", "midclause_pause_per_min",
                                         "boundary_share", "articulation_rate", "phonation_ratio",
                                         "repetition_rate", "duration"],
}
BLACKBOX = {"fine-tuned transformer (DeBERTa)", "speech models (frozen, fine-tuned, speaker kNN)",
            "sentence embedding (mpnet)"}
other = [c for c in USE if c not in {c for cols in GROUPS.values() for c in cols}]
if other:
    GROUPS["other"] = other

gs = {}
for g, cols in GROUPS.items():
    idx = [USE.index(c) for c in cols if c in USE]
    if idx:
        gs[g] = float(np.abs(sv[:, idx].sum(1)).mean())
share = (pd.Series(gs) / sum(gs.values())).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.2))
share.plot(kind="barh", ax=ax, color=["#c0504d" if g in BLACKBOX else "#4f81bd" for g in share.index])
ax.set(xlabel="share of attribution  (mean |SHAP summed within group|)",
       title="What the score is built from — red = learned black-box columns")
plt.tight_layout(); plt.savefig("shap_groups.png", dpi=130); plt.show()
print(share.sort_values(ascending=False).map("{:.0%}".format).to_string())
print(f"\nshare carried by learned black-box columns: "
      f"{share[[g for g in share.index if g in BLACKBOX]].sum():.0%}")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].scatter(y + np.random.normal(0, .05, len(y)), blend_oof, s=14, alpha=.45)
ax[0].plot([1, 5], [1, 5], "k--", lw=1)
ax[0].set(xlabel="true MOS", ylabel="predicted", title="Predicted vs true (out-of-fold)")
ax[1].scatter(blend_oof, y - blend_oof, s=14, alpha=.45); ax[1].axhline(0, color="k", lw=1)
ax[1].set(xlabel="predicted", ylabel="residual", title="Residuals")
pd.DataFrame({"y": y, "e": np.abs(y - blend_oof)}).groupby("y").e.mean().plot(kind="bar", ax=ax[2])
ax[2].set(title="MAE by true score — where not to trust it", xlabel="true MOS")
plt.tight_layout(); plt.savefig("residuals.png", dpi=130); plt.show()


# ---- 6. report cards ----------------------------------------------------------
# Every line uses an explainable feature only, so it is something a candidate
# could hear and act on. The score itself comes from the full model.
pred_of = {("train", f): p for f, p in zip(T.filename, blend_oof)}
pred_of.update({("test", f): p for f, p in zip(TE.filename, blend_test)})
pct = lambda col, v: 100 * float((T[col] < v).mean())


def report_card(split, fn):
    r = F[(F.split == split) & (F.filename == fn)].iloc[0]
    print(f"\n{split}/{fn}   predicted grammar score {pred_of[(split, fn)]:.2f} / 5")
    if "cola_frac_bad" in r.index and pd.notna(r.cola_frac_bad):
        print(f"  - {r.cola_frac_bad:.0%} of sentences were judged ungrammatical by the acceptability model")
    print(f"  - the grammar-correcting transcriber rewrote about {min(r.repair_wer_nofiller, 1):.0%} "
          f"of the words (more rewriting than {pct('repair_wer_nofiller', r.repair_wer_nofiller):.0f}% "
          f"of training speakers)")
    print(f"  - vocabulary range higher than {pct('text_ttr', r.text_ttr):.0f}% of training speakers")
    print(f"  - {'no completed sentences were detected' if r.no_punct else 'completed sentences were detected'}")
    print(f'  - transcript: "{W[(split, fn)]["text"][:200]}..."')


order = np.argsort(blend_test)
for i in (order[0], order[len(order) // 2], order[-1]):
    report_card("test", TE.filename[i])


# ---- 7. submission ------------------------------------------------------------
# Built from test.csv's filename list; sample_submission.csv is stale (204 rows,
# IDs 13-1329, only 25 shared with the 216 in test.csv).
sub = test[["filename"]].merge(pd.DataFrame({"filename": TE.filename, "label": blend_test}),
                               on="filename", how="left")
if sub.label.isna().any():
    print(f"!! {sub.label.isna().sum()} test files had no prediction — filling with train mean")
    sub.label = sub.label.fillna(y.mean())
sub.to_csv("submission.csv", index=False)
print(f"\nsubmission.csv: {len(sub)} rows  min {sub.label.min():.2f}  "
      f"max {sub.label.max():.2f}  mean {sub.label.mean():.2f}")

## 10 · Findings

*Numbers in this section are from version 3 (blend 0.51 out-of-fold, leaderboard 0.386). Version 4 reached 0.375 on the leaderboard; version 5's numbers are printed by §6–§9.*

**Where the signal was.** Hand-built features took the model from 1.01 to 0.68. Every gain after that came from learned representations: a frozen sentence encoder (0.68 → 0.63), a fine-tuned transformer that on its own beats all 21 handcrafted features, and — largest of all — a frozen speech model that never sees the transcript (0.64 → 0.56 on the same stack). After the first few features, counting things was the wrong instrument, for the transcript and the audio alike.

**The most accurate predictor of these grammar ratings is substantially acoustic.** `wavlm_pred` pools a speech model's representations over time, which discards word order: it is a weak instrument for sentence structure and a strong one for how a person sounds. Yet it predicts the grammar rating better than a fine-tuned model reading the transcript (0.56 against 0.63), and it carries 45% of the attribution in the final model. The likeliest reading is that the human scores are partly a judgement of how the speaker sounds — a halo effect, a familiar problem in human-rated speaking tests — and the model has learned it, because it models the raters faithfully. That runs against this project's own requirement to be accent-friendly. Excluding the column gives a less accent-exposed model at a measured price of about 0.07 RMSE on the same feature stack. It is kept here and stated plainly, rather than dropped silently or left unexamined.

**The interpretability trade-off is real and measured.** The ten explainable core features trailed the full set by 0.035 at the start, by 0.11 once the transformer was added, and by 0.16 now. Learned black-box columns carry 68% of the final model's attribution; the explainable features carry 32%. The report cards use only explainable features, so they say something a candidate can act on — but they explain less than a third of what drives the score they sit under.

**Fairness exposure.** Two channels in the final model are accent-sensitive. One is the audio embedding above. The other is `repair_wer`, which is largely a measure of how recognisable someone's speech is: it correlates −0.94 with the CTC model's own confidence, and controlling for that confidence its association with the label falls from −0.55 to roughly −0.2. `train/audio_606.wav` is the concrete case: clean audio (33 dB dynamic range, no clipping), rated by a human, on which CTC recovered 15 of roughly 100 words. CoLA, which sees only normalised text, correlates −0.39 with `repair_wer` — partial corroboration that a share of it is grammar. Before any score from this system were used on a real person, it would need per-accent error analysis.

**Counting silences was the wrong instrument — not the audio.** Mid-clause pausing has the opposite sign to the hypothesis: designed as a negative indicator, it measures +0.15. Pausing mid-clause while building a complex sentence is what a capable speaker does; someone producing short memorised patterns has nothing to pause inside — rubric level 1. The hand-built fluency group alone scores 0.95 against a 1.01 baseline, which once suggested the audio carried little. The speech embedding shows otherwise: the audio carries a great deal, and counting pauses doesn't capture it.

**The test set is a shifted population.** Adversarial AUC ≈0.75, traced to recording length: test files average 48.8 s against 55.7 s in train. `duration` is among the four largest SHAP effects, with short recordings pushed down, and the mean test prediction (3.26) sits 0.22 below the training mean (3.48). The training data cannot say whether test speakers are weaker or their recordings were simply cut shorter; a submission without `duration` would. Leaderboard RMSE has run at about three quarters of CV RMSE across three versions (0.480/0.637, 0.462/0.594, 0.386/0.514), so CV has been a reliable guide to direction, and repeat submissions of one pipeline scored within 0.002 of each other.

**Where not to trust it.** Only 4 training samples fall below 2.0, and the MAE-by-true-score plot puts error at about 0.6 for true 2.0 against 0.3–0.4 through the mid-range. The pull toward the middle is what a calibrated predictor with r = 0.86 does — the blend's RMSE already equals the best any linear rescaling could reach — so it cannot be fixed by post-processing. Separating weak speakers better needs better features.

**Ceiling.** 74% of labels are integers and 26% half-steps, consistent with two raters averaged. That pattern alone cannot pin the noise floor. And because leaderboard RMSE runs at about three quarters of CV here, leaderboard figures — including the public leader's 0.31 — are not on the same scale as the CV numbers in this notebook.

**The 37 zero-label rows are not grammar scores.** IDs 5037–5073, contiguous and disjoint from every other file (0–784), all exactly 0.0, with a near-empty gap at 1.0–1.5 that no real score distribution would have. `audio_5055` is speech under heavy static. A degraded-audio sentinel; it inflates the baseline from 1.01 to 1.24. Dropped from training. Test IDs are 0–215 and never approach 5000.

**Filenames are not unique across splits.** test.csv is `audio_0` … `audio_215`, perfectly contiguous, and 212 of those names also appear in train.csv — independent numbering per folder, not leakage. Everything is keyed on `(split, filename)`. `sample_submission.csv` is stale (204 rows, IDs 13–1329), so the submission is built from test.csv.